<p class="lead">Este notebook cierra el curso, ahora en R. Las diapositivas de clase son un mapa; este texto es el territorio. Las Unidades 1–8 presentaron métodos; esta unidad pregunta qué se hace con ellos en salud, cómo se hace con honestidad y qué obligaciones conlleva hacerlo. Sus cuatro subunidades —minería de datos, reconocimiento de imágenes médicas, sistemas de diagnóstico médico, ética profesional— son cuatro miradas a un mismo objeto: el sistema de reconocimiento de patrones desplegado. Su centro es un <strong>proyecto integrador</strong> (Sección 4.4): un modelo diagnóstico completo sobre los datos de cáncer de mama de Wisconsin, construido con la metodología del Suplemento S1, calibrado, evaluado con una curva de decisión, auditado en cuanto a equidad y explicado.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Describir KDD y CRISP-DM, nombrar las cuatro familias de tareas de la minería de datos y ubicar cada método de las Unidades 2–7 en una de ellas.</li>
<li>Definir soporte, confianza y lift; ejecutar Apriori a mano; explicar por qué minar muchas reglas produce reglas espurias y cómo una distribución nula por permutación y una corrección por comparaciones múltiples las ponen en evidencia.</li>
<li>Usar un isolation forest y una bolsa de palabras (bag of words), e indicar los límites de ambos.</li>
<li>Clasificar una tarea de imágenes; enumerar lo que hace distintas a las imágenes médicas; calcular Dice e IoU; leer críticamente la base de evidencia.</li>
<li>Seguir el diagnóstico asistido por computador desde MYCIN e INTERNIST-1 hasta el apoyo a la decisión; aplicar razones de verosimilitud y el nomograma de Fagan.</li>
<li>Construir, calibrar y evaluar un modelo diagnóstico con honestidad —validación cruzada anidada, calibración, curva de decisión, umbral basado en costos, verificación por subgrupos, model card— y nombrar TRIPOD+AI, DECIDE-AI y CONSORT-AI.</li>
<li>Definir los tres criterios de equidad y el resultado de imposibilidad; auditar y explicar un modelo; describir los datos sintéticos, el aprendizaje federado y la privacidad diferencial; nombrar las normas chilenas e internacionales; aplicar una lista de verificación para el despliegue.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Léelo una vez de forma lineal, ejecutando cada bloque de código (chunk), y luego vuelve a la Sección 4.4 y cambia una cosa a la vez. Las cajas azules contienen ideas clave; las naranjas, trampas; las moradas, vínculos con otras unidades o con BGM324; las verdes, ejercicios. Solo se usan R base, <code>stats</code>, <code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>randomForest</code>, <code>cluster</code>, <code>nnet</code>, <code>pROC</code> y <code>ggplot2</code>; todos los datos vienen con R base o con el paquete <code>mlbench</code>. Se asume el vocabulario del Suplemento S1. En un entorno de ejecución de R recién iniciado en Google Colab, ejecuta una vez <code>install.packages(c("e1071","randomForest","rpart.plot","cluster","mlbench","pROC"))</code>. Donde la edición en Python de esta unidad usaba <code>IsolationForest</code> de scikit-learn, <code>mlxtend</code>/Apriori, un vectorizador de bolsa de palabras y un bucle de aprendizaje federado, la edición en R implementa cada uno desde cero en R base, como prefiere el curso cuando un método resulta más instructivo que una llamada a un paquete.</p></div>



In [ ]:
# Bibliotecas usadas en todo el notebook. Las semillas están fijas para que cada número citado en el texto sea reproducible.
suppressPackageStartupMessages({
  library(MASS); library(class); library(e1071); library(rpart); library(rpart.plot)
  library(randomForest); library(cluster); library(nnet); library(pROC); library(mlbench); library(ggplot2)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"; PURPLE <- "#6a4c93"
cat(R.version.string, "| e1071", as.character(packageVersion("e1071")),
    "| randomForest", as.character(packageVersion("randomForest")),
    "| pROC", as.character(packageVersion("pROC")), "\n")



La salida registra la versión de R y las versiones de los tres paquetes de modelado en los que se apoya el proyecto integrador, para que un lector pueda reproducir cada figura de más abajo.<sup class="c"><a href="#r36">[36]</a></sup>

## 1. Dónde ha llegado el curso

Cada unidad agregó una etapa o un método al pipeline de reconocimiento de patrones de la Unidad 1. La figura vuelve a dibujar ese pipeline con las unidades asociadas a la etapa a la que sirven y con las preocupaciones transversales —evaluación, ética, despliegue— como el marco que lo rodea.



In [ ]:
stages <- list(
  c("Data and\nrepresentation","U1 feature vectors,\ndata types; S0 toolkit", BLUE),
  c("Unsupervised\nstructure","U5 k-means, hierarchical,\nPCA, ICA", BLUE),
  c("Supervised\nlearning","U2 regression, first classifier\nU3 Bayes, SVM, k-NN\nU4 trees, random forest\nU6 neural networks\nU8 image classification", ORANGE),
  c("Optimisation\nand search","U7 genetic algorithms\n(and gradient descent, U2/U6)", GREEN),
  c("Decision and\ndeployment","U9 diagnostic systems,\nthresholds, model cards", PURPLE))
plot.new(); plot.window(xlim=c(0,10), ylim=c(0,6)); 
for (i in seq_along(stages)) {
  s <- stages[[i]]; x <- 0.3 + (i-1)*1.95; col <- s[3]
  rect(x, 2.0, x+1.7, 4.6, border=col, col=adjustcolor(col, 0.12), lwd=1.5)
  text(x+0.85, 4.35, s[1], col=col, font=2, cex=0.8)
  text(x+0.85, 3.0, s[2], cex=0.62)
  if (i < length(stages)) arrows(x+1.72, 3.3, x+1.93, 3.3, length=0.08, lwd=1.2)
}
rect(0.15, 0.7, 9.85, 1.6, col="#f1f1f1", border=GREY)
text(5, 1.15, "Evaluation (Supplement S1): split, cross-validation, confusion matrix, ROC-AUC, leakage - used at every stage", cex=0.72)
rect(0.15, 5.0, 9.85, 5.9, col="#f1f1f1", border=GREY)
text(5, 5.45, "Data mining process (2)  -  Ethics, fairness, privacy, governance (U1 and 5)  -  Reporting standards (4.6)", cex=0.72)
title("The course as one pipeline: the stages and the units that supply them", cex.main=0.95)



<p class="caption"><b>Figura 1.</b> El curso como un solo pipeline: las cinco etapas con la unidad que estudió cada una, y las preocupaciones que se aplican en todas las etapas. La Sección 4.4 recorre la figura completa una vez sobre un solo conjunto de datos.</p>

Tres observaciones organizan la unidad. El pipeline se ejecuta muchas veces, no una sola —de ahí los modelos de proceso de la Sección 2—. Los métodos de las Unidades 2–7 son herramientas para distintas <em>tareas</em> (Sección 2.2). Y las Unidades 1–8 evaluaron los modelos frente a una etiqueta, mientras que esta unidad los evalúa frente a una <em>decisión</em> —ordenar una biopsia o no—, razón por la cual la calibración, las curvas de decisión y los costos dominan la Sección 4.

<div class="box bridge"><h4>Laboratorio 6 y Taller 6</h4><p>Para el Laboratorio 6, el proyecto integrador de la Sección 4.4 es la plantilla y la lista de verificación de la Sección 5.6 es la pauta de evaluación. Para el Taller 6, la Sección 6 enumera quince sistemas reales con su nivel de evidencia, y la Sección 3.3 explica cómo juzgarlo.</p></div>

## 2. Minería de datos (9.1)

### 2.1 Definición y el proceso KDD

Fayyad, Piatetsky-Shapiro y Smyth definieron el <em>descubrimiento de conocimiento en bases de datos</em> (knowledge discovery in databases, KDD) como "el proceso no trivial de identificar patrones válidos, novedosos, potencialmente útiles y, en última instancia, comprensibles en los datos"; la <strong>minería de datos</strong> es el paso en el que un algoritmo los extrae.<sup class="c"><a href="#r1">[1]</a></sup> Cada adjetivo es un requisito —válido en datos nuevos (S1), no conocido de antemano, relevante para la decisión, verificable— y la definición se anticipa en dos décadas a la Sección 5.

KDD recorre selección → preprocesamiento → transformación (Unidades 1, 5) → minería de datos (Unidades 2–7) → interpretación y evaluación, con bucles de retorno. La industria formalizó el ciclo en 2000 como <strong>CRISP-DM</strong>: comprensión del negocio, comprensión de los datos, preparación de los datos, modelado, evaluación, despliegue. Una revisión de 2021 encontró que sigue siendo el modelo de proceso más usado, por lo general con el despliegue omitido o apenas esbozado —con las consecuencias de la Sección 4.7—.<sup class="c"><a href="#r2">[2]</a></sup> Ambos modelos dicen lo mismo: la mayor parte del trabajo ocurre antes y después del algoritmo.



In [ ]:
kdd <- c("Selection","Pre-processing","Transformation","Data mining","Interpretation /\nevaluation")
crisp <- c("Business\nunderstanding","Data\nunderstanding","Data\npreparation","Modelling","Evaluation","Deployment")
par(mfrow=c(2,1), mar=c(1,1,2,1))
for (cfg in list(list(kdd,"KDD process (1996)",BLUE), list(crisp,"CRISP-DM (2000)",ORANGE))) {
  steps <- cfg[[1]]; nm <- cfg[[2]]; col <- cfg[[3]]
  plot.new(); plot.window(xlim=c(0,10), ylim=c(-0.6,2.0)); w <- 9.2/length(steps)
  for (i in seq_along(steps)) { x <- 0.4 + (i-1)*w
    rect(x, 0.5, x+w*0.82, 1.5, col=adjustcolor(col,0.15), border=col)
    text(x+w*0.41, 1.0, steps[i], cex=0.72)
    if (i < length(steps)) arrows(x+w*0.82, 1.0, x+w, 1.0, length=0.06, lwd=1) }
  arrows(0.4+(length(steps)-2)*w+w*0.4, 0.35, 0.6, 0.35, length=0.06, lwd=0.9, col=GREY)
  text(0.3, 1.8, nm, col=col, font=2, cex=0.85, adj=0)
}



<p class="caption"><b>Figura 2.</b> KDD y CRISP-DM. El paso algorítmico es una caja entre cinco o seis; las flechas grises son los bucles de retroalimentación. Solo CRISP-DM nombra el despliegue.</p>

### 2.2 Las cuatro familias de tareas, y dónde encajan las Unidades 2–7

Las tareas de minería de datos se agrupan por <em>pregunta</em>, no por algoritmo, y la primera decisión en cualquier proyecto es qué pregunta se plantea.

| Familia | Pregunta | Salida | Métodos en este curso |
|---|---|---|---|
| **Clasificación** | ¿A qué categoría pertenece este objeto? | Etiqueta de clase o probabilidad | U1 centroide más cercano; U2 regresión logística; U3 Bayes, k-NN, SVM; U4 árboles, bosques; U6 MLP; U8 clasificadores de imágenes |
| **Regresión** | ¿Qué valor de una cantidad continua se espera? | Un número | U2 lineal y polinomial; regresión con k-NN, SVM, árboles y MLP (U3, U4, U6) |
| **Agrupamiento (clustering)** | ¿Qué objetos se parecen entre sí? | Grupos encontrados sin etiquetas | U5 k-means, jerárquico, PCA, ICA |
| **Asociación** | ¿Qué eventos ocurren juntos? | Reglas "si A entonces B" con una fuerza | Reglas de asociación y Apriori — **solo en esta unidad** |

Transversalmente a las familias se ubican la <em>detección de anomalías</em> (Sección 2.7), la <em>optimización</em> (U7) y la <em>minería de texto</em> (Sección 2.8). Ninguna otra unidad aborda las reglas de asociación, por lo que esta sección las trata en profundidad.

<div class="box bridge"><h4>Puente con la bioinformática: la investigación del docente como minería de datos</h4><p>Dos artículos del laboratorio del curso son KDD aplicado a la química. Morales y colaboradores representaron inhibidores de ureasa y compuestos inactivos mediante descriptores fisicoquímicos, entrenaron clasificadores —entre ellos los ensambles de árboles de la Unidad 4— y derivaron <em>criterios de filtrado</em> que un químico puede aplicar antes de la síntesis.<sup class="c"><a href="#r8">[8]</a></sup> González y colaboradores hicieron lo mismo con inhibidores de TRPV1, agregaron un cribado de reposicionamiento de fármacos aprobados y validaron los candidatos mediante docking y simulación de dinámica molecular.<sup class="c"><a href="#r9">[9]</a></sup> Los objetos son moléculas, las características son descriptores, la etiqueta es la actividad y el "despliegue" es la elección de los compuestos que se van a ensayar: los modelos QSAR de BGM324 vistos como reconocimiento de patrones.</p></div>

### 2.3 Reglas de asociación: soporte, confianza, lift

Agrawal, Imieliński y Swami introdujeron las reglas de asociación en 1993 para el problema de la <em>canasta de mercado</em> (market basket).<sup class="c"><a href="#r3">[3]</a></sup> En salud, la canasta es un paciente, los ítems son diagnósticos, medicamentos o alteraciones de laboratorio, y las reglas son patrones de comorbilidad y de prescripción o candidatas a reacciones adversas. Sea \(D\) una base de datos de \(N\) <strong>transacciones</strong>, cada una un subconjunto de los ítems \(I\); un <strong>itemset</strong> es cualquier subconjunto de \(I\) y una <strong>regla</strong> \(A \Rightarrow B\) es una implicación entre itemsets disjuntos. Tres cantidades describen una regla:

<div class="eq">\[\text{supp}(A) = \frac{|\{T \in D : A \subseteq T\}|}{N}, \qquad \text{conf}(A \Rightarrow B) = \frac{\text{supp}(A \cup B)}{\text{supp}(A)}, \qquad \text{lift}(A \Rightarrow B) = \frac{\text{conf}(A \Rightarrow B)}{\text{supp}(B)} = \frac{\text{supp}(A \cup B)}{\text{supp}(A)\,\text{supp}(B)}\]
<span class="where">El soporte estima \(P(A)\) (para una regla, \(\text{supp}(A \cup B)\)); la confianza estima \(P(B \mid A)\); el lift es \(P(A, B)/(P(A)P(B))\) — 1 bajo independencia.</span></div>

La confianza por sí sola engaña: si el 90 % de los pacientes toma paracetamol, "hipertensión ⇒ paracetamol" tiene confianza 0.9 y no dice nada. Brin, Motwani y Silverstein mostraron en 1997 que la confianza puede ordenar las reglas en el orden inverso al de su dependencia y propusieron medir, en cambio, el alejamiento de la independencia —el lift (<em>interés</em>) y, como prueba formal, el estadístico chi-cuadrado de la Sección 2.6—.<sup class="c"><a href="#r5">[5]</a></sup>

### 2.4 El algoritmo Apriori, a mano

Hay \(2^{|I|}\) itemsets, de modo que enumerarlos es imposible. El algoritmo <strong>Apriori</strong> de Agrawal y Srikant (1994) aprovecha la propiedad de <em>clausura descendente</em> (downward closure): <strong>todo subconjunto de un itemset frecuente es frecuente</strong>, por lo que un itemset solo puede ser frecuente si todos sus subconjuntos lo son.<sup class="c"><a href="#r4">[4]</a></sup> Nivel a nivel: (1) conservar los ítems individuales por encima del soporte mínimo, \(L_1\); (2) <strong>unir</strong> (join) pares de \((k-1)\)-itemsets frecuentes que comparten sus primeros \(k-2\) ítems para formar los candidatos \(C_k\); (3) <strong>podar</strong> (prune) los candidatos que tengan un \((k-1)\)-subconjunto que no esté en \(L_{k-1}\); (4) contar los sobrevivientes en una pasada, lo que da \(L_k\); repetir hasta que quede vacío; (5) entregar cada partición \(A \Rightarrow B\) de un itemset frecuente cuya confianza supere un mínimo, junto con su lift.

**Ejemplo resuelto.** Seis pacientes, cinco ítems en el orden HTN < DM < Met < ACEi < Statin (hipertensión, diabetes tipo 2, metformina, inhibidor de la ECA, estatina); soporte mínimo 2 de 6.

| paciente | ítems |
|---|---|
| 1 | HTN, DM, Met, Statin |
| 2 | HTN, ACEi |
| 3 | DM, Met |
| 4 | HTN, DM, Met, ACEi |
| 5 | HTN, Statin |
| 6 | DM, Met, Statin |

**Nivel 1.** HTN 4, DM 4, Met 4, ACEi 2, Statin 3 — los cinco son frecuentes. **Nivel 2.** La unión entrega los \(\binom{5}{2} = 10\) pares (no hay nada que podar, pues todo 1-subconjunto es frecuente); el conteo deja {DM, ACEi}, {Met, ACEi} y {ACEi, Statin} bajo el soporte mínimo. **Nivel 3, unión:** los pares de \(L_2\) con el mismo primer ítem entregan siete candidatos. **Poda:** {HTN, DM, ACEi} contiene el no frecuente {DM, ACEi}; {HTN, Met, ACEi} contiene {Met, ACEi}; {HTN, ACEi, Statin} contiene {ACEi, Statin} — tres podados sin contar. **Conteo** de los cuatro sobrevivientes: {HTN, DM, Met} en los pacientes 1 y 4 (frecuente); {HTN, DM, Statin} y {HTN, Met, Statin} solo en el paciente 1 (no frecuentes); {DM, Met, Statin} en 1 y 6 (frecuente). \(L_3\) tiene dos itemsets que no comparten un prefijo de 2 ítems, de modo que \(C_4\) queda vacío y el algoritmo se detiene. **Reglas:** DM ⇒ Met tiene soporte 4/6 = 0.67, confianza 4/4 = 1.00 y lift 1.00/(4/6) = 1.50; HTN ⇒ Statin tiene confianza 2/4 = 0.50 y lift 0.50/(3/6) = 1.00 — ninguna asociación en absoluto. El bloque de código siguiente reproduce la traza con un Apriori en R base escrito con fines didácticos.



In [ ]:
# Un Apriori mínimo sobre una matriz lógica de transacciones (R base, sin paquetes): unión + poda + conteo, nivel a nivel.
apriori <- function(M, min_support, verbose=FALSE) {
  items <- colnames(M); ord <- function(v) v[order(match(v, items))]
  key <- function(v) paste(ord(v), collapse="|")
  support <- list(); counted <- integer(); L <- list()
  for (j in seq_along(items)) if (mean(M[,j]) >= min_support) L[[length(L)+1]] <- items[j]
  for (c in L) support[[key(c)]] <- mean(M[, c]); k <- 1
  Lkeys <- function(LL) vapply(LL, key, "")
  while (length(L) > 0) {
    k <- k + 1; Lset <- Lkeys(L); C <- list(); seen <- character()
    for (a in L) for (b in L) {                                          # UNIÓN: mismo prefijo de (k-2) ítems
      if (identical(a[-length(a)], b[-length(b)]) && match(a[length(a)], items) < match(b[length(b)], items)) {
        cand <- ord(c(a, b[length(b)]))                                  # PODA: todos los (k-1)-subconjuntos frecuentes
        ok <- all(vapply(seq_along(cand), function(d) key(cand[-d]) %in% Lset, TRUE)); kk <- key(cand)
        if (ok && !(kk %in% seen)) { seen <- c(seen, kk); C[[length(C)+1]] <- cand } } }
    counted[as.character(k)] <- length(C); newL <- list()
    if (length(C)) C <- C[order(vapply(C, function(cc) paste(sprintf("%02d", match(cc, items)), collapse=""), ""))]
    for (cand in C) {                                                    # CONTEO de los sobrevivientes en una pasada
      s <- mean(rowSums(M[, cand, drop=FALSE]) == length(cand))
      if (verbose) cat(sprintf("  L%d {%s}: supp %.2f -> %s\n", k, paste(cand,collapse=", "), s, ifelse(s>=min_support,"frequent","discarded")))
      if (s >= min_support) { support[[key(cand)]] <- s; newL[[length(newL)+1]] <- cand } }
    L <- newL }
  list(support=support, counted=counted, items=items) }

toy <- rbind(c(1,1,1,0,1),c(1,0,0,1,0),c(0,1,1,0,0),c(1,1,1,1,0),c(1,0,0,0,1),c(0,1,1,0,1)) == 1
colnames(toy) <- c("HTN","DM","Met","ACEi","Statin")
res <- apriori(toy, 2/6, verbose=TRUE)
cat("candidates counted per level:", paste(names(res$counted), res$counted, sep="="), "\n")
cat("frequent 3-itemsets:", paste(gsub("\\|", ",", names(res$support)[lengths(strsplit(names(res$support),"\\|"))==3]), collapse=" ; "), "\n")



La traza coincide exactamente con el cálculo a mano: el nivel 2 cuenta los 10 pares (no hay nada que podar), el nivel 3 cuenta solo los 4 sobrevivientes de la poda y los dos 3-itemsets frecuentes son {HTN, DM, Met} y {DM, Met, Statin}; \(C_4\) queda vacío. El bloque siguiente convierte los itemsets frecuentes en reglas.



In [ ]:
# Cada partición A => B de un itemset frecuente, con soporte, confianza y lift.
rules_from <- function(result, min_conf=0.0) {
  support <- result$support; items <- result$items
  ordkey <- function(v) paste(v[order(match(v, items))], collapse="|"); rows <- list()
  for (S in names(support)) {
    SI <- strsplit(S, "\\|")[[1]]; if (length(SI) < 2) next; s <- support[[S]]
    for (r in 1:(length(SI)-1)) { cb <- combn(SI, r)
      for (ci in seq_len(ncol(cb))) { A <- cb[,ci]; B <- setdiff(SI, A)
        conf <- s/support[[ordkey(A)]]
        rows[[length(rows)+1]] <- data.frame(antecedent=paste(A[order(match(A,items))],collapse=" & "),
          consequent=paste(B[order(match(B,items))],collapse=" & "), support=s, confidence=conf,
          lift=conf/support[[ordkey(B)]], stringsAsFactors=FALSE) } } }
  out <- do.call(rbind, rows); out <- out[out$confidence>=min_conf,]; out[order(-out$lift),] }

toy_rules <- rules_from(res)
sel <- toy_rules[toy_rules$antecedent %in% c("DM","HTN") & toy_rules$consequent %in% c("Met","Statin"),]
sel[,c("support","confidence","lift")] <- round(sel[,c("support","confidence","lift")], 2)
print(sel, row.names=FALSE)



Las reglas calculadas confirman los valores obtenidos a mano: DM ⇒ Met tiene soporte 0.67, confianza 1.00 y lift 1.50, mientras que HTN ⇒ Statin tiene confianza 0.50 y lift 1.00 —el mismo soporte que varias otras y ninguna asociación en absoluto—. El mismo código funciona sin cambios sobre la tabla de 1,500 pacientes de más abajo.

<div class="exercise"><h4>Ejercicio 1</h4><p>Baja el soporte mínimo a 1/6 y vuelve a ejecutar ambos bloques: ¿cuántos candidatos de nivel 3 se cuentan ahora y cuántas reglas resultan? Luego agrega un séptimo paciente con los ítems {ACEi, Statin} con soporte 2/6: ¿qué candidato de nivel 3 antes podado se cuenta ahora, y llega a ser frecuente?</p></div>

### 2.5 Apriori sobre una base de datos clínica sintética

Los registros reales no pueden distribuirse junto con un notebook, así que el bloque siguiente genera 1,500 pacientes con siete comorbilidades y ocho medicamentos, sembrando asociaciones plausibles —metformina para los diabéticos, inhibidores de la ECA para los hipertensos, estatinas para ambos, salbutamol para la EPOC, ISRS para la depresión, betabloqueadores para la insuficiencia cardíaca, la obesidad que aumenta la diabetes y la hipertensión que aumenta la insuficiencia cardíaca— y dejando todo lo demás independiente, para poder ver qué reglas recupera el minero y cuáles inventa.



In [ ]:
# Una 'ficha electrónica' sintética de 1,500 pacientes: comorbilidades y medicamentos con asociaciones sembradas.
set.seed(0); n_pat <- 1500
bern <- function(p, size=n_pat) runif(size) < p
htn <- bern(0.45); obes <- bern(0.30); copd <- bern(0.12); dep <- bern(0.15); ckd <- bern(0.08)
dm <- bern(0.25) | (obes & bern(0.25))                      # la obesidad aumenta la probabilidad de diabetes
hf <- bern(0.10) | (htn & bern(0.10))                       # la hipertensión aumenta la probabilidad de insuficiencia cardíaca
prescribed <- function(cond, p_if, p_else) ifelse(cond, bern(p_if), bern(p_else))
records <- data.frame(
  hypertension=htn, type2_diabetes=dm, obesity=obes, COPD=copd, heart_failure=hf, depression=dep, CKD=ckd,
  metformin=prescribed(dm,0.75,0.02), insulin=prescribed(dm,0.25,0.01), ACE_inhibitor=prescribed(htn,0.55,0.05),
  beta_blocker=prescribed(hf,0.70,0.08), statin=prescribed(dm & htn,0.70,0.0) | prescribed(dm | htn,0.35,0.08),
  salbutamol=prescribed(copd,0.65,0.04), SSRI=prescribed(dep,0.60,0.03), aspirin=prescribed(hf | htn,0.30,0.10))
M <- as.matrix(records)
cat(sprintf("%d patients x %d items; items per patient: median %.0f, max %d\n",
    nrow(M), ncol(M), median(rowSums(M)), max(rowSums(M))))
print(round(sort(colMeans(M), decreasing=TRUE), 3))



Los soportes de los ítems van desde la hipertensión, con alrededor de 0.47, hasta la enfermedad renal crónica, con alrededor de 0.07, y un paciente típico presenta tres ítems. Las dependencias sembradas son invisibles en estas frecuencias marginales: viven en la distribución conjunta, que es lo que explora Apriori.



In [ ]:
# Apriori con soporte mínimo 0.05; luego, las reglas con confianza >= 0.5, ordenadas por lift.
t0 <- Sys.time(); res_rec <- apriori(M, min_support=0.05)
by_level <- sapply(1:5, function(k) sum(lengths(strsplit(names(res_rec$support),"\\|"))==k))
cat(sprintf("%.2f s | frequent itemsets per level (1..5): %s | candidates counted: %s\n",
    as.numeric(difftime(Sys.time(),t0,units="secs")), paste(by_level,collapse=" "),
    paste(names(res_rec$counted), res_rec$counted, sep="=", collapse=" ")))
rec_rules <- rules_from(res_rec, min_conf=0.5)
cat(nrow(rec_rules), "rules with confidence >= 0.5; the 8 with the highest lift:\n")
top <- head(rec_rules, 8); top[,c("support","confidence","lift")] <- round(top[,c("support","confidence","lift")], 3)
print(top, row.names=FALSE)



El minero recupera la estructura sembrada en unas pocas centésimas de segundo. Los lifts más altos, de alrededor de 5 a 6, corresponden a los pares fármaco–indicación (depresión ⇔ ISRS, insuficiencia cardíaca ⇔ betabloqueador) y a reglas compuestas sobre hipertensión, diabetes, metformina, inhibidor de la ECA y estatina, cuyos consecuentes de dos ítems son poco frecuentes. Cada asociación aparece en ambas direcciones con igual soporte y lift pero distinta confianza —"SSRI ⇒ depression" tiene más confianza que "depression ⇒ SSRI" porque no todo paciente deprimido recibe tratamiento, pero casi todo usuario de ISRS está deprimido—; el lift es simétrico, la confianza no lo es, y ninguno de los dos es causal. Las reglas sobre itemsets compuestos son en gran medida redundantes con las reglas por pares que las componen.

### 2.6 La trampa: reglas espurias y comparaciones múltiples

Un minero pone a prueba cada par y cada itemset mayor: 105 pares para 15 ítems, más de un millón para 1,500 códigos CIE-10. Algunos tendrán un lift muy por encima de 1 por azar, y el minero, que no tiene hipótesis nula, los informa junto a los reales. El bloque cuantifica esto <em>destruyendo todas las asociaciones</em> —permutando cada columna de forma independiente, lo que conserva las frecuencias de los ítems pero hace que cada ítem sea independiente de todos los demás— y minando la tabla permutada con un soporte mínimo más bajo, donde los pares poco frecuentes son ruidosos.



In [ ]:
# Distribución nula por permutación: barajar cada columna de forma independiente (asociaciones destruidas, frecuencias conservadas) y volver a minar.
pairwise <- function(rt) { r <- rt[!grepl("&", rt$antecedent) & !grepl("&", rt$consequent),]; r[r$antecedent < r$consequent,] }
real_pairs <- pairwise(rules_from(apriori(M, 0.02)))
set.seed(0); null_max <- numeric(); null_lifts <- numeric()
for (rep in 1:50) { sh <- apply(M, 2, sample)
  pr <- pairwise(rules_from(apriori(sh, 0.02))); null_lifts <- c(null_lifts, pr$lift); null_max <- c(null_max, max(pr$lift)) }
cat(sprintf("real table: %d pairwise rules, %d with lift > 1.3, maximum lift %.2f\n",
    nrow(real_pairs), sum(real_pairs$lift > 1.3), max(real_pairs$lift)))
cat(sprintf("50 shuffled tables: maximum pairwise lift per table ranges %.2f-%.2f; %.1f%% of null rules have lift > 1.3\n",
    min(null_max), max(null_max), 100*mean(null_lifts > 1.3)))
bins <- seq(0.3, 2.2, length.out=39)
hist(null_lifts, breaks=bins, freq=FALSE, col=adjustcolor(GREY,0.7), border="white",
     main="Lift of rules with no association behind them (grey) spreads well beyond 1", xlab="lift", cex.main=0.9)
hist(pmin(real_pairs$lift, 2.2), breaks=bins, freq=FALSE, col=adjustcolor(ORANGE,0.6), border="white", add=TRUE)
abline(v=1, lty=2); legend("topright", c("null (shuffled tables)","real table (clipped at 2.2)"),
       fill=c(adjustcolor(GREY,0.7), adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figura 3.</b> Lift de las reglas por pares de 50 copias barajadas de la tabla (gris; toda asociación destruida) frente a la tabla real (naranja; lifts sembrados de hasta alrededor de 5.5, recortados en 2.2). La distribución nula está centrada en 1 pero alcanza alrededor de 1.5–1.7: quien informa cada regla con lift superior a 1.3 informa reglas de la distribución gris.</p>

El remedio es el de la Unidad 3 y del Suplemento S1: tratar cada regla como una prueba de hipótesis —para un par, la prueba chi-cuadrado de independencia sobre su tabla de 2 × 2— y corregir por las \(m\) pruebas realizadas, con <strong>Bonferroni</strong> (\(p < \alpha/m\)) o con la tasa de falsos descubrimientos de Benjamini–Hochberg. El bloque siguiente lo hace para las reglas por pares de la tabla real.



In [ ]:
# Prueba chi-cuadrado de independencia para cada regla por pares, con y sin corrección de Bonferroni.
chi2_p <- function(a, b) suppressWarnings(chisq.test(table(records[[a]], records[[b]]))$p.value)
tested <- real_pairs; tested$p_value <- mapply(chi2_p, tested$antecedent, tested$consequent); m <- nrow(tested)
cat(sprintf("%d pairwise rules tested | p < 0.05: %d | Bonferroni p < 0.05/%d: %d\n",
    m, sum(tested$p_value < 0.05), m, sum(tested$p_value < 0.05/m)))
ov <- tested[tested$lift > 1.1 & tested$lift < 1.6,]; ov <- ov[order(-ov$lift),]
ov$sig_05 <- ov$p_value < 0.05; ov$sig_bonf <- ov$p_value < 0.05/m
ov$support <- round(ov$support,3); ov$lift <- round(ov$lift,3); ov$p_value <- signif(ov$p_value,2)
print(head(ov[,c("antecedent","consequent","support","lift","p_value","sig_05","sig_bonf")], 8), row.names=FALSE)



De 81 reglas por pares, 27 pasan \(p < 0.05\) y 21 sobreviven al umbral más estricto de Bonferroni; las seis que pasan uno y no el otro son exactamente las reglas débiles y limítrofes de la región de solapamiento. Las reglas sembradas (obesidad ⇒ diabetes, hipertensión ⇒ estatina, aspirina ⇒ hipertensión) sobreviven con valores \(p\) que son numéricamente cero. Quedan tres límites: una asociación que sobrevive puede ser trivial; puede estar <em>confundida</em>, como lo está metformina ⇒ obesidad (pasa a través de la diabetes); y ninguna regla es causal —la minería genera hipótesis que los diseños epidemiológicos ponen a prueba—. La farmacovigilancia usa exactamente este pipeline: medidas de desproporcionalidad, tamizaje estadístico, revisión clínica.

<div class="box warn"><h4>El look-elsewhere effect</h4><p>Todo método que <em>busca</em> —reglas, anomalías, subgrupos (Sección 5.2), importancias de características (Sección 5.3)— pone a prueba muchas hipótesis a la vez, y el máximo de muchas estimaciones ruidosas está sesgado hacia arriba, como mostró S1 para la selección de hiperparámetros. Toda cantidad seleccionada por ser la mayor debe volver a estimarse con datos que no participaron en la selección.</p></div>

### 2.7 La detección de anomalías en un bloque de código

La detección de anomalías pregunta <em>¿qué objetos no se parecen al resto?</em> y no necesita etiquetas —es la pregunta para fallas de equipos, reclamos fraudulentos, valores de laboratorio de una enfermedad no reconocida o errores de digitación—. El <strong>isolation forest</strong> de Liu, Ting y Zhou reutiliza los árboles de la Unidad 4: divisiones aleatorias hasta que cada punto queda aislado, y el <em>número de divisiones necesarias</em> es el puntaje —las anomalías, pocas y diferentes, se aíslan rápidamente—.<sup class="c"><a href="#r6">[6]</a></sup> Los paquetes del curso no incluyen un isolation forest en R, así que el bloque implementa uno en unas veinte líneas de R base, lo entrena solo con los casos <em>benignos</em> de Wisconsin, como si la malignidad fuera una condición nunca vista, y pregunta si los casos malignos quedan marcados.



In [ ]:
# Los datos de cáncer de mama de Wisconsin de mlbench: el sustituto en R de load_breast_cancer de scikit-learn.
data(BreastCancer, package="mlbench")
bc <- BreastCancer[,-1]                                  # eliminar la columna Id
bc <- bc[complete.cases(bc),]                            # eliminar las 16 filas con valores faltantes
bc[,1:9] <- lapply(bc[,1:9], function(z) as.numeric(as.character(z)))   # factor ordinal -> numérico
feat <- as.matrix(bc[,1:9]); yc <- as.integer(bc$Class == "malignant")  # yc: 1 = maligno (codificación de 'enfermedad')
cat("rows x features:", nrow(feat), "x", ncol(feat), "| malignant:", sum(yc), "of", length(yc), "\n")



La tabla depurada tiene 683 aspirados descritos por nueve puntajes ordinales de morfología nuclear, 239 de ellos malignos.<sup class="c"><a href="#r37">[37]</a></sup> Tiene nueve características en lugar de las treinta de la copia de scikit-learn, pero es el mismo problema diagnóstico y la misma fuente.



In [ ]:
# Isolation forest desde cero: árboles de divisiones aleatorias; puntaje de anomalía = 2^(-E[longitud del camino]/c(n)).
c_factor <- function(n) if (n > 2) 2*(log(n-1)+0.5772156649) - 2*(n-1)/n else if (n==2) 1 else 0
itree <- function(X, depth, md) { n <- nrow(X)
  if (depth >= md || n <= 1) return(list(leaf=TRUE, size=n))
  f <- sample(ncol(X), 1); rg <- range(X[,f]); if (rg[1]==rg[2]) return(list(leaf=TRUE, size=n))
  sp <- runif(1, rg[1], rg[2]); lft <- X[,f] < sp
  list(leaf=FALSE, f=f, sp=sp, L=itree(X[lft,,drop=FALSE],depth+1,md), R=itree(X[!lft,,drop=FALSE],depth+1,md)) }
path_len <- function(x, nd, d=0) if (nd$leaf) d + c_factor(nd$size) else if (x[nd$f] < nd$sp) path_len(x,nd$L,d+1) else path_len(x,nd$R,d+1)
iforest <- function(X, n_trees=200, ss=256) { X <- as.matrix(X); n <- nrow(X); ss <- min(ss,n); md <- ceiling(log2(ss))
  list(trees=lapply(1:n_trees, function(t) itree(X[sample(n,ss),,drop=FALSE],0,md)), cn=c_factor(ss)) }
score_if <- function(mdl, X) { X <- as.matrix(X); 2^(-rowMeans(sapply(mdl$trees, function(tr) apply(X,1,path_len,nd=tr)))/mdl$cn) }

set.seed(0); benign_only <- feat[yc==0,]
iso <- iforest(benign_only); score <- score_if(iso, feat)
auc_anom <- as.numeric(auc(yc, score, direction="<", quiet=TRUE))
cutoff <- quantile(score_if(iso, benign_only), 0.90)     # punto de operación: marcar lo que sea más anómalo que el 90% de los normales
flag <- score >= cutoff
cat(sprintf("trained on %d benign cases | AUC of the anomaly score for malignancy: %.3f\n", nrow(benign_only), auc_anom))
cat(sprintf("flagged as anomalous: %.1f%% of malignant, %.1f%% of benign\n", 100*mean(flag[yc==1]), 100*mean(flag[yc==0])))
hist(score[yc==0], breaks=30, col=adjustcolor(BLUE,0.7), border="white", xlim=range(score),
     main="Isolation forest: malignant aspirates are isolated faster", xlab="anomaly score", ylab="cases", cex.main=0.9)
hist(score[yc==1], breaks=30, col=adjustcolor(ORANGE,0.7), border="white", add=TRUE)
legend("topright", c("benign (seen in training)","malignant (never seen)"), fill=c(adjustcolor(BLUE,0.7),adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figura 4.</b> Puntajes de anomalía de los 683 aspirados según un isolation forest que vio solo casos benignos. Los casos malignos (naranja) se aíslan con menos divisiones y el puntaje alcanza un AUC de alrededor de 0.99 sin una sola etiqueta maligna. El punto de operación —marcar todo lo que sea más anómalo que el 90 % de los datos normales de entrenamiento— detecta casi todos los casos malignos, mientras que, por construcción, genera una falsa alarma en alrededor del 10 % de los benignos.</p>

Sobre estas nueve características ordinales limpias, el detector separa las clases con más nitidez que la copia de treinta características de scikit-learn (allí el AUC era de alrededor de 0.96, con un solapamiento considerable), pero la lección no cambia: un modelo supervisado sobre los mismos datos alcanza un AUC comparable (el proyecto integrador de la Sección 4.4 llega a alrededor de 0.997), y el precio de no tener etiquetas es que el detector también marca fallas del instrumento y condiciones benignas poco frecuentes. Su papel es el tamizaje de lo inesperado y el control de calidad; su salida es una lista ordenada para un humano, no un diagnóstico: lo <em>diferente</em> no es lo <em>enfermo</em>.

### 2.8 La minería de texto en un párrafo

La mayor parte de la información clínica es texto libre. La manera más simple de llevarla a la matriz de datos de la Unidad 1 es la <strong>bolsa de palabras</strong> (bag of words): cada documento se convierte en un vector de conteos de palabras sobre un vocabulario compartido, ignorando el orden; luego vienen Bayes ingenuo, SVM o agrupamiento. La representación no sabe que "no fever" niega "fever" ni que "MI" es "myocardial infarction" (infarto de miocardio); el procesamiento de lenguaje natural clínico consiste en gran medida en reparar esos defectos —detección de negaciones, expansión de abreviaturas, mapeo a SNOMED CT, embeddings contextuales de modelos de lenguaje—. Una revisión de 263 estudios de extracción de información clínica encontró que los métodos de bolsa de palabras y los basados en reglas seguían dominando, que la mayoría de los sistemas eran de una sola institución y que pocos tenían validación externa.<sup class="c"><a href="#r7">[7]</a></sup> El bloque muestra la idea con seis oraciones sintéticas y un Bayes ingenuo multinomial escrito desde cero.



In [ ]:
# Bolsa de palabras: seis notas clínicas sintéticas se convierten en una matriz de conteos de 6 x |vocabulario|; un Bayes ingenuo multinomial aprende de ella.
notes <- c("patient reports chest pain on exertion and shortness of breath",
           "productive cough for three weeks with fever and night sweats",
           "chest pain radiating to left arm, diaphoresis, nausea",
           "persistent cough, wheeze, history of smoking, no fever",
           "exertional chest tightness relieved by rest", "cough with fever and pleuritic chest pain")
label <- c("cardiac","respiratory","cardiac","respiratory","cardiac","respiratory")
stop_en <- c("on","and","for","with","to","of","by","the","a","an","no","history")
tok <- function(s) { w <- unlist(strsplit(tolower(gsub("[^a-z ]"," ",s)), "\\s+")); w[nchar(w)>0 & !(w %in% stop_en)] }
toks <- lapply(notes, tok); vocab <- sort(unique(unlist(toks)))
bow <- t(sapply(toks, function(t) table(factor(t, levels=vocab)))); rownames(bow) <- paste0("note ", seq_along(notes))
mnb_fit <- function(X,y,alpha=1) { cl <- unique(y)
  list(cl=cl, lp=setNames(log(sapply(cl,function(c) mean(y==c))),cl),
       ll=sapply(cl, function(c){ cnt <- colSums(X[y==c,,drop=FALSE])+alpha; log(cnt/sum(cnt)) })) }
mnb_pred <- function(m,X) { sc <- sweep(X %*% m$ll, 2, m$lp, "+")
  list(pred=m$cl[apply(sc,1,which.max)], prob=t(apply(sc,1,function(r){e<-exp(r-max(r)); e/sum(e)}))) }
nb <- mnb_fit(bow, label); new_note <- "sudden chest pain with sweating"
pr <- mnb_pred(nb, matrix(table(factor(tok(new_note),levels=vocab)), nrow=1))
cat(sprintf("vocabulary of %d words (stop-words removed); matrix %d x %d\n", length(vocab), nrow(bow), ncol(bow)))
cat("new note:", new_note, "->", pr$pred, "| P(cardiac) =", round(pr$prob[1, which(nb$cl=="cardiac")], 2), "\n")
print(bow[, c("chest","pain","cough","fever","exertion","sweats")])



Seis oraciones se convirtieron en una matriz de conteos sobre 27 palabras, y el Bayes ingenuo multinomial de la Unidad 3 aprendió que "chest" y "pain" apuntan a una nota cardíaca y que "cough" y "fever" apuntan a una respiratoria, de modo que la nota nueva se clasifica como cardíaca con una probabilidad de alrededor de 0.73. La falla ya es visible: la nota 6 contiene "chest pain" (pleurítico, respiratorio) y la nota 4 un "fever" negado, y sin embargo ambos cuentan como presentes. El clasificador es un juguete; la representación es la que usó a gran escala la primera generación del NLP clínico, con estos límites.

<div class="exercise"><h4>Ejercicio 2</h4><p>Agrega dos notas que usen negación ("no chest pain", "denies fever") con etiquetas apropiadas y vuelve a ejecutar: ¿cambia la predicción para la nota nueva? Luego forma características de pares de palabras concatenando tokens adyacentes (un vectorizador de bigramas) para que "no fever" se convierta en una sola característica, e inspecciona cuántas columnas más gana el vocabulario. ¿Cuánto cuesta esta corrección en número de características?</p></div>

## 3. Reconocimiento de imágenes médicas (9.2)

La Unidad 8 hizo el trabajo técnico de esta subunidad; esta sección es una síntesis: las tareas, por qué las imágenes médicas son difíciles, cómo se mide una segmentación, qué dice la evidencia.

### 3.1 La taxonomía de tareas

| Tarea | Entrada → salida | Ejemplo clínico | Métrica típica |
|---|---|---|---|
| **Clasificación** | Imagen completa → etiqueta o probabilidad | Retinopatía diabética referible a partir de una fotografía de fondo de ojo; lesión cutánea maligna vs benigna | Sensibilidad, especificidad, ROC-AUC (S1) |
| **Detección** | Imagen → cajas delimitadoras o puntos | Nódulos pulmonares en TC; microcalcificaciones en una mamografía | Sensibilidad a un número dado de falsos positivos por imagen |
| **Segmentación** | Imagen → una etiqueta por píxel/vóxel | Volumen tumoral en RM; contorno ventricular; núcleos en histología | Dice, IoU (Sección 3.4) |
| **Registro** | Dos imágenes → transformación de alineamiento | Exámenes pre y postoperatorios; etiquetado con atlas; corrección de movimiento | Error de registro del objetivo (mm) |
| **Reconstrucción** | Datos crudos o degradados → imagen | RM submuestreada; eliminación de ruido en TC de baja dosis | SNR, similitud estructural |

Los datos de Wisconsin se sitúan aguas arriba de las cinco: sus características de morfología nuclear se extrajeron de núcleos previamente <em>segmentados</em> en una imagen de citología.

### 3.2 Qué hace diferentes a las imágenes médicas

El aprendizaje profundo para imágenes se desarrolló sobre ImageNet: 1.2 millones de fotografías con etiquetas que cualquiera puede aportar. Las imágenes médicas difieren en todos los aspectos, y cada diferencia es un modo de falla de un método transferido ingenuamente: **conjuntos de datos pequeños** (las redes se sobreajustan; el transfer learning y el aumento de datos, Unidad 8, son remedios parciales); **etiquetas escasas y ruidosas** (el tiempo de un radiólogo, una biopsia, un desenlace años después; el desacuerdo entre lectores le da a la verdad de referencia su propia tasa de error); **tres y cuatro dimensiones** (muchos datos por caso, pocos casos); **desbalance de clases** (prevalencia en tamizaje inferior al 1 %, donde la exactitud carece de sentido); **domain shift** (fabricante, protocolo, población — Sección 3.5); **señal sutil y local** (unas pocas docenas de píxeles entre megapíxeles); **alto riesgo** (el clínico debe poder verificar la salida — Sección 5.3).

### 3.3 La base de evidencia

Liu y colaboradores agruparon 82 estudios que comparaban el aprendizaje profundo con profesionales de la salud sobre las mismas imágenes: sensibilidad agrupada de alrededor del 87 % y especificidad de alrededor del 93 % para los modelos, frente a 86 % y 91 % para los clínicos —equivalencia en promedio—, pero solo 25 estudios validaron fuera de la muestra y solo 14 compararon a ambos sobre la misma muestra externa.<sup class="c"><a href="#r10">[10]</a></sup> Nagendran y colaboradores examinaron el diseño: de 81 estudios no aleatorizados, pocos eran prospectivos, la mayoría comparaba el modelo con un puñado de clínicos que leían imágenes aisladas, los datos y el código rara vez estaban disponibles, los resúmenes a menudo exageraban y existían solo dos ensayos aleatorizados.<sup class="c"><a href="#r11">[11]</a></sup> El problema no es que los modelos no funcionen, sino que el artículo habitualmente no puede decirnos si funcionan.

<div class="box key"><h4>Cómo leer una afirmación de desempeño</h4><p>Pregunta, en este orden: (1) ¿validación externa, o un conjunto hold-out del mismo origen? (2) ¿se comparó a los clínicos en condiciones realistas, o leyendo imágenes aisladas? (3) ¿a qué prevalencia, y qué significa la especificidad en falsos positivos a la prevalencia de uso? (4) ¿la verdad de referencia es una biopsia o un desenlace, o la opinión de otro lector? (5) ¿un estudio prospectivo, y un ensayo aleatorizado de la <em>decisión</em>? La mayoría de los sistemas publicados no cumple (1); casi todos no cumplen (5).</p></div>

### 3.4 Métricas de segmentación: Dice e IoU

Una máscara predicha \(S\) se compara con una referencia \(G\). La exactitud por píxel es inútil —un tumor que ocupa el 2 % de la imagen le da una exactitud del 98 % a una máscara vacía—, así que se usan medidas de solapamiento. El <strong>coeficiente de Dice</strong>, llamado así por el ecólogo L. R. Dice, quien lo propuso en 1945 para la coocurrencia de especies, y la <strong>intersección sobre la unión</strong> (IoU, índice de Jaccard) son

<div class="eq">\[\text{Dice}(S, G) = \frac{2\,|S \cap G|}{|S| + |G|}, \qquad \text{IoU}(S, G) = \frac{|S \cap G|}{|S \cup G|}, \qquad \text{Dice} = \frac{2\,\text{IoU}}{1 + \text{IoU}}\]
<span class="where">Ambos van de 0 a 1 y ordenan las segmentaciones de manera idéntica; Dice es el puntaje F1 del Suplemento S1 aplicado a píxeles.</span></div>



In [ ]:
# Dice e IoU sobre un par de máscaras sintéticas: una 'lesión' circular (verdad) y una predicción desplazada y agrandada.
g <- expand.grid(x=0:63, y=0:63)
truth <- (g$x-30)^2 + (g$y-32)^2 <= 10^2                  # referencia: radio 10 centrada en (30, 32)
pred  <- (g$x-34)^2 + (g$y-30)^2 <= 11^2                  # predicción: desplazada en (4, -2), radio 11
inter <- sum(truth & pred); union <- sum(truth | pred)
dice <- 2*inter/(sum(truth)+sum(pred)); iou <- inter/union
cat(sprintf("|G| = %d, |S| = %d, |S & G| = %d, |S | G| = %d\n", sum(truth), sum(pred), inter, union))
cat(sprintf("Dice = %.3f, IoU = %.3f, 2*IoU/(1+IoU) = %.3f  |  pixel accuracy = %.3f\n", dice, iou, 2*iou/(1+iou), mean(truth==pred)))
mk <- function(v) matrix(v, 64, 64)
ov <- array(1, c(64,64,3))
TT <- mk(truth); PP <- mk(pred)
ov[,,1][TT & PP] <- 0.3; ov[,,2][TT & PP] <- 0.6; ov[,,3][TT & PP] <- 0.3
ov[,,1][TT & !PP] <- 0.2; ov[,,2][TT & !PP] <- 0.4; ov[,,3][TT & !PP] <- 0.8
ov[,,1][!TT & PP] <- 0.9; ov[,,2][!TT & PP] <- 0.5; ov[,,3][!TT & PP] <- 0.2
par(mfrow=c(1,3), mar=c(1,1,2,1))
image(mk(truth), col=c("white",BLUE), axes=FALSE, main="reference mask G")
image(mk(pred),  col=c("white",ORANGE), axes=FALSE, main="predicted mask S")
plot(as.raster(aperm(ov, c(2,1,3)))); title(sprintf("overlap: Dice %.2f, IoU %.2f", dice, iou))



<p class="caption"><b>Figura 5.</b> Una máscara de referencia, una predicción desplazada en cuatro píxeles y algo más grande de lo debido, y su solapamiento (verde en ambas, azul omitido, naranja sobresegmentado). La exactitud por píxel es de alrededor de 0.95 porque la mayor parte de la imagen es fondo, y sin embargo Dice es alrededor de 0.73 e IoU alrededor de 0.57. El Dice entre expertos es de alrededor de 0.9 para órganos bien definidos y de 0.7 o menos para lesiones difusas, lo que acota lo que se puede demostrar que cualquier modelo logra.</p>

### 3.5 Un pipeline de imágenes pequeño y completo, como recordatorio

R no tiene un conjunto de datos de dígitos, así que el bloque <em>simula</em> uno: seis patrones de 8 × 8 (un anillo, una barra, un siete, una cruz, una L y una X), cada uno dibujado 170 veces con amplitud aleatoria y ruido por píxel, con los píxeles del borde dejados a propósito cerca de cero. Luego se ejecuta sobre ellos todo el pipeline de la Figura 1 —división estratificada, estandarización, PCA (Unidad 5), una máquina de vectores de soporte (Unidad 3) y un perceptrón multicapa (Unidad 6)— como ensayo para la Sección 4.4.



In [ ]:
# Imágenes simuladas de 'dígitos' de 8x8 (seis clases, borde en cero); un recordatorio de que R no tiene load_digits.
templates <- list(
  zero = c(0,0,0,0,0,0,0,0, 0,0,1,1,1,1,0,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,1,0,0,0,0,1,0, 0,0,1,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  one  = c(0,0,0,0,0,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  seven= c(0,0,0,0,0,0,0,0, 0,1,1,1,1,1,0,0, 0,0,0,0,0,1,0,0, 0,0,0,0,1,0,0,0, 0,0,0,1,0,0,0,0, 0,0,1,0,0,0,0,0, 0,0,1,0,0,0,0,0, 0,0,0,0,0,0,0,0),
  plus = c(0,0,0,0,0,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,1,1,1,1,1,1,0, 0,1,1,1,1,1,1,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,0,0,0,0,0,0,0),
  ell  = c(0,0,0,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,0,0,0,0,0, 0,1,1,1,1,1,0,0, 0,0,0,0,0,0,0,0),
  ex   = c(0,0,0,0,0,0,0,0, 0,1,0,0,0,0,1,0, 0,0,1,0,0,1,0,0, 0,0,0,1,1,0,0,0, 0,0,0,1,1,0,0,0, 0,0,1,0,0,1,0,0, 0,1,0,0,0,0,1,0, 0,0,0,0,0,0,0,0))
set.seed(0); per <- 170
X <- do.call(rbind, lapply(templates, function(b) t(sapply(1:per, function(i)
  pmin(pmax(b*12*runif(1,0.55,1.15) + rnorm(64,0,2.4), 0), 16)))))
yd <- factor(rep(names(templates), each=per), levels=names(templates))
idx <- unlist(lapply(levels(yd), function(c){ i<-which(yd==c); sample(i, round(0.7*length(i))) }))
Xtr <- X[idx,]; ytr <- yd[idx]; Xte <- X[-idx,]; yte <- yd[-idx]
mu <- colMeans(Xtr); sdv <- apply(Xtr,2,sd); Z <- scale(Xtr,mu,sdv); Zte <- scale(Xte,mu,sdv)
pc <- prcomp(Z, center=FALSE, scale.=FALSE); k <- 20; Ptr <- pc$x[,1:k]; Pte <- Zte %*% pc$rotation[,1:k]
set.seed(1); folds <- sample(rep(1:5, length.out=nrow(Ptr)))
cv_acc <- mean(sapply(1:5, function(f){ m <- svm(Ptr[folds!=f,], ytr[folds!=f], kernel="radial", cost=10, gamma=0.01)
  mean(predict(m, Ptr[folds==f,])==ytr[folds==f]) }))
svm_fit <- svm(Ptr, ytr, kernel="radial", cost=10, gamma=0.01)
mlp_fit <- nnet(Z, class.ind(ytr), size=16, softmax=TRUE, maxit=250, trace=FALSE, MaxNWts=3000)
cat(sprintf("%d simulated images, %d classes | PCA(%d)+SVM: CV acc %.3f, test acc %.3f | MLP: test acc %.3f\n",
    nrow(X), nlevels(yd), k, cv_acc, mean(predict(svm_fit,Pte)==yte), mean(levels(ytr)[max.col(predict(mlp_fit,Zte))]==yte)))



La tarea simulada es lo bastante limpia como para que ambos pipelines clasifiquen las imágenes reservadas de forma prácticamente perfecta, y —el punto del bloque— la estimación por validación cruzada de 5 particiones sobre el conjunto de entrenamiento coincide con el puntaje de prueba, como exige el Suplemento S1. Los hiperparámetros se eligieron dentro del conjunto de entrenamiento, el conjunto de prueba se usó una vez y ambos modelos se compararon sobre la misma división. El bloque siguiente muestra contra qué <em>no</em> protege esa coincidencia.



In [ ]:
# Domain shift: los mismos modelos aplicados a imágenes de prueba de un 'escáner distinto' (más brillantes, de menor contraste, desenfocadas).
other_scanner <- function(Xm, gain=0.7, offset=4, blur=TRUE) { out <- Xm*gain + offset
  if (blur) out <- t(apply(out, 1, function(img){ m <- matrix(img,8,8,byrow=TRUE); mb <- m
    for (i in 1:8) for (j in 1:8) mb[i,j] <- mean(m[max(1,i-1):min(8,i+1), max(1,j-1):min(8,j+1)]); as.vector(t(mb)) }))
  pmin(pmax(out,0),16) }
Xte_sh <- other_scanner(Xte); Zte_sh <- scale(Xte_sh,mu,sdv); Pte_sh <- Zte_sh %*% pc$rotation[,1:k]
acc_svm_sh <- mean(predict(svm_fit, Pte_sh)==yte); acc_mlp_sh <- mean(levels(ytr)[max.col(predict(mlp_fit, Zte_sh))]==yte)
cat(sprintf("accuracy  same scanner -> different scanner:  SVM %.3f -> %.3f | MLP %.3f -> %.3f\n",
    mean(predict(svm_fit,Pte)==yte), acc_svm_sh, mean(levels(ytr)[max.col(predict(mlp_fit,Zte))]==yte), acc_mlp_sh))
js <- sapply(levels(yte), function(cl) which(yte==cl)[1])                   # un ejemplo por clase
par(mfrow=c(2,6), mar=c(1,1,1.6,0.3))
for (j in js) { image(matrix(Xte[j,],8,8,byrow=TRUE)[,8:1], col=gray.colors(16,1,0), axes=FALSE); title(paste("true", yte[j]), cex.main=0.8) }
for (j in js) { p <- predict(svm_fit, Pte_sh[j,,drop=FALSE])
  image(matrix(Xte_sh[j,],8,8,byrow=TRUE)[,8:1], col=gray.colors(16,1,0), axes=FALSE)
  title(paste("->", p), cex.main=0.8, col.main=ifelse(p==yte[j],GREEN,ORANGE)) }



<p class="caption"><b>Figura 6.</b> Seis imágenes de prueba tal como se adquirieron (arriba) y tras un cambio simulado de escáner —menor contraste, un desplazamiento de brillo, un leve desenfoque (abajo)—, con la predicción de PCA + SVM para cada una (verde correcta, naranja incorrecta). Las imágenes siguen siendo legibles para una persona, y sin embargo la exactitud cae de prácticamente perfecta a alrededor de 0.15 para la SVM (nivel de azar para seis clases) y a alrededor de 0.28 para el MLP.</p>

El mecanismo es típico: los píxeles del borde de las imágenes de entrenamiento casi siempre valen cero, de modo que el estandarizador aprendió para ellos una media y una varianza cercanas a cero; el desplazamiento de brillo los convierte en valores estandarizados grandes que ninguna imagen de entrenamiento tuvo jamás, y ambos modelos extrapolan. El cambio que más daña no es el más visible para una persona (el desenfoque), sino el que aleja más los datos de la distribución de entrenamiento en la representación del modelo. La evaluación interna honesta fue honesta respecto de la población equivocada, y los dos modelos se vieron afectados de manera distinta, así que la robustez se mide, no se supone. Remedios: validación externa en el sitio de despliegue, adquisición armonizada, aumento de datos con las variaciones esperadas (Unidad 8), monitoreo después del despliegue (Sección 5.6).

<div class="exercise"><h4>Ejercicio 3</h4><p>Varía el cambio: fija <code>blur=FALSE</code>; luego <code>gain=1.0</code> solo con el desplazamiento de brillo; luego solo el desenfoque. ¿Qué componente daña más a cada modelo? Luego vuelve a ajustar ambos pipelines con imágenes de entrenamiento aumentadas con sus propias copias desplazadas (<code>rbind(Xtr, other_scanner(Xtr))</code>) y vuelve a evaluar en ambos conjuntos de prueba. ¿Restaura el aumento de datos la exactitud con el cambio, y cuánto cuesta en el conjunto original?</p></div>

## 4. Sistemas de diagnóstico médico (9.3)

### 4.1 Una breve historia

La Unidad 1 describió <strong>MYCIN</strong> (Stanford, 1972–1976), el consultor de antibióticos basado en reglas. Su contemporáneo <strong>INTERNIST-1</strong> (Pittsburgh, desde 1974) abarcaba unas 500 enfermedades y 3,550 manifestaciones, vinculadas mediante una <em>evoking strength</em> y una <em>frecuencia</em>. En su evaluación de 1982 sobre 19 casos de conferencias clinicopatológicas tuvo un desempeño comparable al de los clínicos del hospital e inferior al de los expertos que discutían los casos, y sus autores dijeron con claridad que no estaba listo para el uso clínico: no podía razonar sobre el tiempo, la anatomía ni la causalidad, y no podía explicarse a sí mismo.<sup class="c"><a href="#r12">[12]</a></sup> Su base de conocimiento se convirtió en <strong>QMR</strong>; <strong>DXplain</strong> todavía genera diagnósticos diferenciales en hospitales universitarios; y <strong>HELP</strong> (LDS Hospital, Salt Lake City) fue el primer sistema de información hospitalaria con lógica de decisión incorporada —alertas disparadas a partir de los datos de laboratorio y farmacia a medida que llegaban—, lo que mostró que el sistema diagnóstico útil es un monitor que habla cuando es necesario, no un consultor al que hay que preguntarle. Estos sistemas no se difundieron por las razones que la Unidad 1 dio para MYCIN —conocimiento frágil adquirido a mano, ningún lugar en el flujo de trabajo, ningún responsable—, y lo que sobrevivió fue la arquitectura de HELP y los propios registros.

### 4.2 Los sistemas de apoyo a la decisión clínica hoy

Un <strong>sistema de apoyo a la decisión clínica</strong> (clinical decision support system, CDSS) presenta información específica del paciente a un clínico en el punto de atención: **alertas y recordatorios** (verificación de interacciones y dosis —basados en reglas, plagados de <em>fatiga de alertas</em>, con tasas de anulación superiores al 90 %—); **puntajes de riesgo y modelos de predicción**; **priorización (triage)** de listas de trabajo según la urgencia predicha; **IA diagnóstica en imágenes** como segundo lector, como tamizaje previo o, en un caso autorizado (Sección 6), como decisión autónoma; y **conjuntos de órdenes y apoyo a la documentación**, que desde 2023 incluyen modelos de lenguaje que redactan notas.

La guía para implementadores de Osheroff y colaboradores destiló los <strong>five rights de los CDSS</strong>: la <em>información</em> correcta, a la <em>persona</em> correcta (quien puede actuar —no necesariamente el médico—), en el <em>formato</em> correcto, por el <em>canal</em> correcto, en el <em>momento correcto del flujo de trabajo</em> —la lista de verificación para la fase de despliegue que la mayoría de los proyectos se salta—. La etiqueta <em>model facts</em> de Sendak y colaboradores —una página que indica qué predice un modelo, en quiénes se validó, cómo debe y cómo no debe usarse, a quién llamar— es el instrumento para la información correcta en el formato correcto.<sup class="c"><a href="#r31">[31]</a></sup> Rajkomar, Dean y Kohane plantean el punto complementario: los problemas difíciles junto a la cama del paciente son el flujo de datos, la integración en el flujo de trabajo y la evaluación del efecto sobre la atención, no el algoritmo de aprendizaje.<sup class="c"><a href="#r30">[30]</a></sup>

### 4.3 Razonamiento diagnóstico bayesiano y el nomograma de Fagan

Todo sistema diagnóstico actualiza una creencia con evidencia (Unidad 3). La práctica clínica usa la forma del teorema de Bayes en términos de <em>odds</em> (chances), en la que el resultado de una prueba actúa por multiplicación: con odds pre-test \(O_0 = p_0/(1-p_0)\) y las <strong>razones de verosimilitud</strong> (likelihood ratios) de la prueba,

<div class="eq">\[\text{LR}^{+} = \frac{\text{sensitivity}}{1 - \text{specificity}}, \qquad \text{LR}^{-} = \frac{1 - \text{sensitivity}}{\text{specificity}}, \qquad O_{\text{post}} = O_0 \times \text{LR}, \qquad p_{\text{post}} = \frac{O_{\text{post}}}{1 + O_{\text{post}}}\]
<span class="where">Sensibilidad = especificidad = 0.90 da \(\text{LR}^{+} = 9\) y \(\text{LR}^{-} = 0.11\). En escala logarítmica la actualización es aditiva, \(\log O_{\text{post}} = \log O_0 + \log \text{LR}\), razón por la cual el nomograma de Fagan es una línea recta.</span></div>

El nomograma de Fagan de 1975 pone las tres cantidades en ejes logarítmicos paralelos, de modo que una línea recta desde la probabilidad pre-test a través de la razón de verosimilitud da la probabilidad post-test.<sup class="c"><a href="#r13">[13]</a></sup> Hace visible el hecho central del diagnóstico —<em>el mismo resultado positivo significa cosas distintas a distintas probabilidades pre-test</em>—, aquí para \(\text{LR}^{+} = 9\) en un contexto de tamizaje (\(p_0 = 0.05\)) y en uno de derivación (\(p_0 = 0.37\), la prevalencia en los datos de Wisconsin).



In [ ]:
# Nomograma de Fagan: probabilidad pre-test, razón de verosimilitud y probabilidad post-test en ejes paralelos de log-odds.
post_test <- function(p0, lr) { o <- p0/(1-p0)*lr; o/(1+o) }
logit <- function(p) log(p/(1-p)); sens <- 0.90; spec <- 0.90
lr_pos <- sens/(1-spec); lr_neg <- (1-sens)/spec
cases <- data.frame(p0=c(0.05,0.37,0.05,0.37), result=c("positive","positive","negative","negative"), LR=c(lr_pos,lr_pos,lr_neg,lr_neg))
cases$post <- mapply(post_test, cases$p0, cases$LR)
cat(sprintf("sensitivity %.2f, specificity %.2f -> LR+ = %.2f, LR- = %.3f\n", sens, spec, lr_pos, lr_neg))
print(transform(cases, post=round(post,3)), row.names=FALSE)
p_ticks <- c(.001,.002,.005,.01,.02,.05,.1,.2,.3,.5,.7,.8,.9,.95,.98,.99,.995,.998,.999)
lr_ticks <- c(.001,.002,.005,.01,.02,.05,.1,.2,.5,1,2,5,10,20,50,100,200,500,1000)
plot.new(); plot.window(xlim=c(-0.5,2.5), ylim=c(-7.6,9.0))
for (p in p_ticks) { text(-0.06, -logit(p), formatC(p*100,format="g"), adj=1, cex=0.6); text(2.06, logit(p), formatC(p*100,format="g"), adj=0, cex=0.6) }
for (lr in lr_ticks) text(1.05, log(lr)/2, formatC(lr,format="g"), adj=0, cex=0.6)
for (x in c(0,1,2)) segments(x,-7,x,7, lwd=1)
cols <- c(ORANGE,BLUE,ORANGE,BLUE)
for (i in 1:4) segments(0, -logit(cases$p0[i]), 2, logit(cases$post[i]), col=cols[i], lwd=1.6, lty=ifelse(cases$result[i]=="positive",1,3))
for (xt in list(c(0,"pre-test\nprobability (%)"), c(1,"likelihood\nratio"), c(2,"post-test\nprobability (%)"))) text(as.numeric(xt[1]), 8.4, xt[2], cex=0.7, font=2)
legend("bottom", sprintf("pre %.0f%%, %s (LR %.2f) -> post %.1f%%", cases$p0*100, cases$result, cases$LR, cases$post*100),
       col=cols, lty=ifelse(cases$result=="positive",1,3), lwd=1.6, bty="n", cex=0.65)
title("Fagan's nomogram for a test with sensitivity = specificity = 0.90", cex.main=0.9)



<p class="caption"><b>Figura 7.</b> El nomograma de Fagan calculado en código, en escalas de log-odds para que el teorema de Bayes sea una línea recta. El mismo resultado positivo (continuo, LR⁺ = 9) lleva a un paciente de tamizaje del 5 % a alrededor del 32 % y a un paciente derivado del 37 % a alrededor del 84 %; el mismo resultado negativo (punteado, LR⁻ = 0.11) los lleva a alrededor del 0.6 % y del 6 %. Una prueba "90 % exacta" deja a dos de cada tres pacientes de tamizaje con resultado positivo libres de enfermedad.</p>

<div class="box key"><h4>Qué significa esto para un sistema que aprende</h4><p>La salida de un clasificador es una probabilidad post-test <em>a la prevalencia de su conjunto de entrenamiento</em>. Desplegado donde la prevalencia es distinta, sus probabilidades están equivocadas exactamente en la razón de las odds a priori, aunque sus razones de verosimilitud no cambien. Por eso la Sección 4.4 informa la prevalencia de su conjunto de evaluación, la calibración debe volver a verificarse en cada sitio y un valor predictivo positivo citado sin una prevalencia carece de sentido.</p></div>

### 4.4 El proyecto integrador: un modelo diagnóstico completo y honesto

El resto de esta sección construye un modelo diagnóstico para los datos de cáncer de mama de Wisconsin como debe construirse un modelo destinado a pacientes, siguiendo el Suplemento S1 y los estándares de la Sección 4.6:

1. **Problema y datos.** Clase positiva = maligno. División estratificada 75/25; el conjunto de prueba se toca una sola vez, al final.
2. **Modelos candidatos.** Máquina de vectores de soporte con kernel RBF, bosque aleatorio, perceptrón multicapa (Unidades 3, 4, 6), cada uno estandarizado de modo que ninguna estadística de escalamiento se filtre entre particiones.
3. **Validación cruzada anidada**: búsqueda interna de hiperparámetros con 3 particiones, estimación externa con 5 particiones; se selecciona la familia con el AUC externo más alto.
4. **Calibración** mediante Platt scaling, ajustada sobre predicciones de entrenamiento obtenidas por validación cruzada.
5. **Evaluación en el conjunto de prueba**: ROC-AUC, curva de confiabilidad y puntaje de Brier, curva de decisión.
6. **Umbral** a partir del costo relativo de un cáncer no detectado y de una biopsia innecesaria, elegido sobre predicciones de entrenamiento obtenidas por validación cruzada.
7. **Verificación por subgrupos** y **model card**.



In [ ]:
# Pasos 1-2: división estratificada, tres familias candidatas como ajustes estandarizados, y grillas de búsqueda.
auc_of <- function(truth, prob) as.numeric(auc(truth, prob, levels=c("ben","mal"), direction="<", quiet=TRUE))
std_fit <- function(Xt) list(mu=colMeans(Xt), sd=apply(Xt,2,sd)); std_ap <- function(s,Xt) scale(Xt, s$mu, s$sd)
make_folds <- function(y,k,seed){ set.seed(seed); f<-integer(length(y)); for(c in levels(y)){i<-which(y==c); f[i]<-sample(rep(1:k,length.out=length(i)))}; f }
fit_predict <- function(fam, par, Xt, yt, Xn){ s<-std_fit(Xt); Zt<-std_ap(s,Xt); Zn<-std_ap(s,Xn)
  if (fam=="svm")      attr(predict(svm(Zt,yt,kernel="radial",cost=par$C,gamma=par$gamma,probability=TRUE), Zn, probability=TRUE),"probabilities")[,"mal"]
  else if (fam=="rf")  predict(randomForest(Zt,yt,ntree=150,mtry=par$mtry,nodesize=par$nodesize), Zn, type="prob")[,"mal"]
  else                 predict(nnet(Zt,class.ind(yt),size=par$size,decay=par$decay,maxit=400,trace=FALSE,MaxNWts=3000), Zn)[,"mal"] }
grids <- list(svm=expand.grid(C=c(0.3,1,3,10), gamma=c(0.01,0.03,0.1)),
              rf=expand.grid(mtry=c(2,3), nodesize=c(1,3)), mlp=expand.grid(size=c(4,8), decay=c(0.01,0.1)))
fam_label <- c(svm="SVM, RBF (Unit 3)", rf="Random forest (Unit 4)", mlp="MLP (Unit 6)")
tune <- function(fam, Xt, yt, inner_k=3, seed=0){ g<-grids[[fam]]; f<-make_folds(yt,inner_k,seed); aucs<-numeric(nrow(g))
  for (gi in 1:nrow(g)) aucs[gi] <- mean(sapply(1:inner_k, function(k) auc_of(yt[f==k], fit_predict(fam, as.list(g[gi,]), Xt[f!=k,], yt[f!=k], Xt[f==k,]))))
  list(best=as.list(g[which.max(aucs),]), auc=max(aucs)) }
X <- as.matrix(bc[,1:9]); y <- factor(ifelse(bc$Class=="malignant","mal","ben"), levels=c("ben","mal"))
set.seed(0); tr <- unlist(lapply(levels(y), function(c){ i<-which(y==c); sample(i, round(0.75*length(i))) }))
Xtr <- X[tr,]; ytr <- y[tr]; Xte <- X[-tr,]; yte <- y[-tr]
cat(sprintf("training set: %d cases (%.1f%% malignant) | test set: %d cases (%.1f%% malignant), locked away\n",
    length(ytr), 100*mean(ytr=="mal"), length(yte), 100*mean(yte=="mal")))
cat("hyper-parameter combinations per family:", paste(names(grids), sapply(grids, nrow), sep="="), "\n")



Los 683 aspirados se dividen en 512 casos de entrenamiento y 171 de prueba, cada conjunto con un 35 % de malignos gracias a la estratificación. (La copia de scikit-learn tenía 569 casos; la copia de mlbench es más grande, por lo que estos conteos difieren de los de la edición en Python aunque el problema es idéntico). El conjunto de prueba queda ahora apartado y no se consulta hasta el paso 5.



In [ ]:
# Paso 3: validación cruzada anidada (ajuste interno con 3 particiones, estimación externa con 5 particiones) de las tres familias.
outer <- make_folds(ytr, 5, 0); nested <- data.frame()
for (fam in c("svm","rf","mlp")) { set.seed(0); aucs <- accs <- numeric(5); first_par <- NULL
  for (k in 1:5) { best <- tune(fam, Xtr[outer!=k,], ytr[outer!=k], 3, 0); if (k==1) first_par <- best$best
    pr <- fit_predict(fam, best$best, Xtr[outer!=k,], ytr[outer!=k], Xtr[outer==k,])
    aucs[k] <- auc_of(ytr[outer==k], pr); accs[k] <- mean((pr>=0.5)==(ytr[outer==k]=="mal")) }
  nested <- rbind(nested, data.frame(family=fam_label[fam], outer_AUC=mean(aucs), AUC_sd=sd(aucs), outer_acc=mean(accs),
    best_fold1=paste(names(first_par), unlist(first_par), sep="=", collapse=", "))) }
chosen_fam <- c("svm","rf","mlp")[which.max(nested$outer_AUC)]
cat("selected family (highest outer AUC):", fam_label[chosen_fam], "\n")
nested2 <- nested; nested2[,2:4] <- round(nested2[,2:4], 4); print(nested2, row.names=FALSE)



Las tres familias alcanzan un AUC en las particiones externas superior a 0.98 y difieren en unas pocas milésimas —menos que la desviación estándar entre particiones—, de modo que la tabla no ofrece fundamentos sólidos para preferir una; el código selecciona la media más alta, en esta ejecución la máquina de vectores de soporte, y la selección es en sí misma una elección hecha sobre datos de entrenamiento. El perceptrón multicapa se ajusta con <code>nnet</code>, cuya red de una sola capa oculta y su compañera <code>lda</code> están documentadas en <em>Modern Applied Statistics with S</em> de Venables y Ripley.<sup class="c"><a href="#r41">[41]</a></sup> La Unidad 3 hizo la misma comparación sin un conjunto de prueba apartado; solo los números presentes estiman el desempeño en casos nuevos, porque el conjunto de prueba aún no se ha usado.



In [ ]:
# Pasos 4-5a: reajustar la familia seleccionada con todos los datos de entrenamiento, calibrar con Platt scaling y evaluar una vez en el conjunto de prueba.
best_overall <- tune(chosen_fam, Xtr, ytr, 3, 0)
oof <- numeric(length(ytr)); ocv <- make_folds(ytr,5,0)                    # puntajes de entrenamiento out-of-fold (cross_val_predict)
for (k in 1:5) oof[ocv==k] <- fit_predict(chosen_fam, best_overall$best, Xtr[ocv!=k,], ytr[ocv!=k], Xtr[ocv==k,])
clamp <- function(p) pmin(pmax(p,1e-6),1-1e-6)
platt <- glm((ytr=="mal") ~ qlogis(clamp(oof)), family=binomial); co <- coef(platt)   # sigmoide de Platt sobre el logit del puntaje
raw_te <- fit_predict(chosen_fam, best_overall$best, Xtr, ytr, Xte)
p_te <- plogis(co[1] + co[2]*qlogis(clamp(raw_te)))
auc_te <- auc_of(yte, p_te); brier <- mean((p_te - (yte=="mal"))^2); prev <- mean(yte=="mal")
cm <- table(truth=yte, pred=factor(ifelse(p_te>=0.5,"mal","ben"), levels=c("ben","mal")))
tn<-cm[1,1]; fp<-cm[1,2]; fn<-cm[2,1]; tp<-cm[2,2]
cat(sprintf("tuned parameters: %s\n", paste(names(best_overall$best), unlist(best_overall$best), sep="=", collapse=", ")))
cat(sprintf("test set (%d cases): ROC-AUC %.3f | Brier %.3f (%.3f for a constant prediction at prevalence)\n", length(yte), auc_te, brier, prev*(1-prev)))
cat(sprintf("at threshold 0.5: TP %d, FN %d, FP %d, TN %d -> sensitivity %.3f, specificity %.3f, PPV %.3f\n",
    tp, fn, fp, tn, tp/(tp+fn), tn/(tn+fp), tp/(tp+fp)))



Ajustado una vez con 512 casos de entrenamiento y evaluado una vez con 171 casos de prueba, el modelo alcanza un AUC de prueba de alrededor de 0.997 y un puntaje de Brier de alrededor de 0.013 (frente a 0.228 si se predice la prevalencia para todos); con umbral 0.5 omite 2 de 60 casos malignos y genera 1 falsa alarma entre 111 benignos.<sup class="c"><a href="#r38">[38]</a>,<a href="#r39">[39]</a>,<a href="#r40">[40]</a></sup> Estos son los números que informa un artículo. Los bloques siguientes preguntan lo que un artículo habitualmente no pregunta: ¿están calibradas las probabilidades?, ¿es útil el modelo en los umbrales que usan los clínicos?, ¿y es 0.5 el umbral correcto?



In [ ]:
# Paso 5b: calibración - ¿coinciden las probabilidades predichas con las frecuencias observadas? Comparación con el bosque aleatorio sin calibrar.
p_rf <- fit_predict("rf", tune("rf", Xtr, ytr, 3, 0)$best, Xtr, ytr, Xte); brier_rf <- mean((p_rf-(yte=="mal"))^2)
reliability <- function(yv, p, nb=6){ q <- unique(quantile(p, seq(0,1,length.out=nb+1))); q[1]<- -Inf; q[length(q)]<-Inf
  b <- cut(p, q, include.lowest=TRUE); na.omit(data.frame(mp=tapply(p,b,mean), obs=tapply(as.integer(yv=="mal"),b,mean))) }
par(mfrow=c(1,2), mar=c(4,4,2,1))
rc <- reliability(yte, p_te); rr <- reliability(yte, p_rf)
plot(rc$mp, rc$obs, type="o", pch=19, col=BLUE, xlim=0:1, ylim=0:1, xlab="mean predicted probability (bin)",
     ylab="observed fraction malignant", main="Reliability curve (test set, 6 quantile bins)", cex.main=0.85)
lines(rr$mp, rr$obs, type="o", pch=19, col=ORANGE); abline(0,1,lty=2)
legend("topleft", c(sprintf("calibrated SVM (Brier %.3f)",brier), sprintf("random forest (Brier %.3f)",brier_rf), "perfect"),
       col=c(BLUE,ORANGE,"black"), lty=c(1,1,2), pch=c(19,19,NA), bty="n", cex=0.7)
hist(p_te[yte=="ben"], breaks=seq(0,1,0.04), col=adjustcolor(BLUE,0.7), border="white", xlab="predicted probability of malignancy",
     ylab="test cases", main="Predicted probabilities by true class", cex.main=0.85)
hist(p_te[yte=="mal"], breaks=seq(0,1,0.04), col=adjustcolor(ORANGE,0.7), border="white", add=TRUE)
legend("top", c("benign","malignant"), fill=c(adjustcolor(BLUE,0.7),adjustcolor(ORANGE,0.6)), bty="n", cex=0.8)



<p class="caption"><b>Figura 8.</b> Izquierda: curvas de confiabilidad en el conjunto de prueba —casos agrupados en intervalos según la probabilidad predicha, predicción media frente a fracción observada de malignos—; un modelo calibrado se ubica sobre la diagonal. Derecha: probabilidades predichas por el modelo calibrado según la clase verdadera; la mayoría de los casos se ubica cerca de 0 o de 1. El puntaje de Brier (leyenda) es la diferencia cuadrática media entre la probabilidad predicha y el desenlace.</p>

La discriminación y la calibración son propiedades distintas: un modelo puede ordenar los casos a la perfección y, aun así, informar probabilidades sistemáticamente demasiado tímidas o demasiado audaces. El Platt scaling ajusta una regresión logística desde el puntaje al desenlace sobre predicciones de entrenamiento obtenidas por validación cruzada y corrige esa distorsión. Sobre estas nueve características limpias, el bosque aleatorio sin calibrar ya está cerca de la diagonal (su puntaje de Brier es apenas superior al 0.013 del modelo calibrado), de modo que aquí la corrección es pequeña —una salvedad que los datos de treinta características de la edición en Python hacían mayor—. La calibración sigue importando porque el umbral del paso 6 y el beneficio neto del paso 5 viven <em>en la escala de probabilidad</em>.



In [ ]:
# Paso 5c: decision curve analysis (Vickers y Elkin 2006) en R base - beneficio neto frente a la probabilidad umbral.
net_benefit <- function(yv, p, th) { yy <- as.integer(yv=="mal"); n <- length(yy)
  sapply(th, function(t){ pr <- p>=t; sum(pr & yy==1)/n - sum(pr & yy==0)/n * t/(1-t) }) }
th <- seq(0.01,0.79,0.01); nb_model <- net_benefit(yte,p_te,th); nb_rf <- net_benefit(yte,p_rf,th)
nb_all <- prev - (1-prev)*th/(1-th)
dt <- c(0.05,0.10,0.20,0.33,0.50)
dca <- data.frame(threshold=dt, model=net_benefit(yte,p_te,dt), biopsy_all=prev-(1-prev)*dt/(1-dt), biopsy_none=0)
cat("net benefit (true positives per 100 patients, net of harm from false positives):\n"); print(round(dca*100,1), row.names=FALSE)
plot(th, nb_model, type="l", col=BLUE, lwd=2, ylim=c(-0.05, prev+0.05), xlab="threshold probability (where a clinician would act)",
     ylab="net benefit", main="Decision curve on the test set", cex.main=0.9)
lines(th, nb_rf, col=ORANGE, lwd=1.2); lines(th, nb_all, col=GREY, lwd=1.2, lty=2); abline(h=0, lwd=1)
legend("topright", c("calibrated SVM","random forest","biopsy everyone","biopsy no one"),
       col=c(BLUE,ORANGE,GREY,"black"), lty=c(1,1,2,1), lwd=c(2,1.2,1.2,1), bty="n", cex=0.75)



<p class="caption"><b>Figura 9.</b> Curvas de decisión en el conjunto de prueba. El eje horizontal es la <em>probabilidad umbral</em> \(p_t\) a la que un clínico o un paciente optaría por la biopsia —cómo se pondera un cáncer no detectado frente a un procedimiento innecesario—; el eje vertical es el beneficio neto. "Biopsy everyone" (biopsiar a todos, línea discontinua) y "biopsy no one" (no biopsiar a nadie, línea cero) acotan la región en la que un modelo es útil; el modelo calibrado se ubica por encima de ambas en todo el rango plausible.</p>

El decision curve analysis de Vickers y Elkin responde lo que el AUC y la calibración no responden: <em>¿decidir con este modelo llevaría a mejores decisiones que no usarlo?</em><sup class="c"><a href="#r14">[14]</a></sup> Su único supuesto es que un clínico que actúa a la probabilidad \(p_t\) valora un verdadero positivo como \((1-p_t)/p_t\) falsos positivos —quien biopsia al 10 % acepta nueve biopsias innecesarias por cada cáncer encontrado—, de modo que

<div class="eq">\[\text{NB}(p_t) = \frac{\text{TP}(p_t)}{n} - \frac{\text{FP}(p_t)}{n} \cdot \frac{p_t}{1 - p_t}\]
<span class="where">en verdaderos positivos por paciente. "Biopsiar a todos" da \(\text{NB} = \pi - (1-\pi)\,p_t/(1-p_t)\), con \(\pi\) la prevalencia; "no biopsiar a nadie" da 0. Un modelo es útil en \(p_t\) solo si su curva se ubica por encima de ambas.</span></div>

La tabla sobre la figura lee la curva en cinco umbrales: con \(p_t = 0.10\), el modelo produce alrededor de 35 verdaderos positivos por cada 100 pacientes, descontado el daño, frente a alrededor de 28 al biopsiar a todos. Cerca de \(p_t \to 0\), toda estrategia converge a "biopsiar a todos": un clínico que va a biopsiar a cualquier probabilidad no necesita un modelo.

### 4.5 Elegir el umbral según el costo clínico

El umbral por defecto de 0.5 trata un cáncer no detectado y una biopsia innecesaria como igualmente costosos. Si una neoplasia maligna no detectada es \(c\) veces peor, el costo esperado de un umbral es \(c \cdot \text{FN} + \text{FP}\) y el óptimo de la teoría de decisión (Unidad 3) es actuar cuando \(p > 1/(1+c)\). El bloque elige el umbral de manera empírica a partir de <em>predicciones de entrenamiento obtenidas por validación cruzada</em> e informa la matriz de confusión del conjunto de prueba y las razones de verosimilitud en ese umbral.



In [ ]:
# Paso 6: umbral según el costo clínico (FN = c x FP), elegido sobre predicciones de ENTRENAMIENTO obtenidas por validación cruzada y aplicado una vez al conjunto de prueba.
expected_cost <- function(yv,p,t,c){ pr<-p>=t; c*sum(!pr & yv=="mal") + sum(pr & yv=="ben") }
grid_t <- seq(0.02,0.94,0.01); thr_rows <- data.frame()
for (cc in c(1,3,5,10)) { tstar <- grid_t[which.min(sapply(grid_t, function(t) expected_cost(ytr, oof, t, cc)))]
  cmx <- table(yte, factor(ifelse(p_te>=tstar,"mal","ben"), levels=c("ben","mal")))
  tn<-cmx[1,1];fp<-cmx[1,2];fn<-cmx[2,1];tp<-cmx[2,2]; se<-tp/(tp+fn); sp<-tn/(tn+fp)
  thr_rows <- rbind(thr_rows, data.frame(cost_ratio=cc, theoretical=1/(1+cc), threshold=tstar, test_FN=fn, test_FP=fp,
    sensitivity=se, specificity=sp, LRpos=ifelse(sp<1, se/(1-sp), Inf), LRneg=(1-se)/sp)) }
T_STAR <- thr_rows$threshold[thr_rows$cost_ratio==5]
cat(sprintf("operating threshold adopted (c = 5): %.2f\n", T_STAR))
tr2 <- thr_rows; num <- c("theoretical","threshold","sensitivity","specificity","LRpos","LRneg"); tr2[,num] <- round(tr2[,num],3)
print(tr2, row.names=FALSE)



A medida que aumenta la razón de costos, el umbral debería bajar y la sensibilidad subir. Aquí las probabilidades calibradas están separadas con tanta nitidez —la mayoría de los casos queda a menos de una centésima de 0 o de 1— que la superficie de costo es plana y las cuatro razones de costos seleccionan el mismo umbral de operación, de alrededor de 0.14, que ya lleva a cero los falsos negativos del conjunto de prueba a cambio de dos falsos positivos. Por lo tanto, el umbral empírico sigue a \(1/(1+c)\) solo de manera aproximada, una versión más marcada de la misma salvedad que señalaba la edición en Python. Adoptamos \(c = 5\). Las razones de verosimilitud en ese umbral se trasladan directamente a la Figura 7: la probabilidad post-test tras una salida positiva o negativa depende de la prevalencia en el lugar donde se usa el modelo.

La razón de costos no le corresponde elegirla solo al ingeniero: codifica un juicio clínico y ético que pertenece a los clínicos, a los pacientes y a la institución. La obligación del ingeniero es exponer la elección y mostrar sus consecuencias, como lo hacen la tabla y la curva de decisión.

### 4.6 Verificación por subgrupos, model card y estándares de reporte

Un modelo bueno en promedio puede ser malo en un subgrupo —un rango de edad, un escáner, un hospital—. Los datos de Wisconsin no contienen variables demográficas, así que el bloque simula la verificación más simple: casos de prueba asignados al azar a dos "hospitales", donde el desempeño debería ser igual por construcción. La Sección 5.2 la repite con una variable de subgrupo <em>correlacionada</em> con el desenlace.



In [ ]:
# Paso 7a: desempeño por subgrupo. Aquí el subgrupo es una etiqueta de 'hospital' aleatoria, así que las diferencias son ruido de muestreo.
subgroup_report <- function(yv,p,group,t) do.call(rbind, lapply(sort(unique(group)), function(gg){ m<-group==gg
  cmx <- table(factor(yv[m],levels=c("ben","mal")), factor(ifelse(p[m]>=t,"mal","ben"),levels=c("ben","mal")))
  tn<-cmx[1,1];fp<-cmx[1,2];fn<-cmx[2,1];tp<-cmx[2,2]
  data.frame(group=gg, n=sum(m), prevalence=mean(yv[m]=="mal"), AUC=auc_of(yv[m],p[m]),
             sensitivity=tp/(tp+fn), specificity=tn/(tn+fp), flagged=mean(p[m]>=t)) }))
set.seed(0); hospital <- sample(c("Hospital A","Hospital B"), length(yte), replace=TRUE, prob=c(0.6,0.4))
sg <- subgroup_report(yte, p_te, hospital, T_STAR); sg[,3:7] <- round(sg[,3:7],3); print(sg, row.names=FALSE)



Los dos hospitales tienen la misma sensibilidad y difieren en alrededor de media centésima en especificidad, con aproximadamente 65–105 casos cada uno. Primera lección: con \(n \approx 65\)–105 por grupo, un falso negativo adicional mueve la sensibilidad en dos a cuatro puntos, de modo que tales diferencias son compatibles con un desempeño igual, y deben informarse intervalos de confianza, no estimaciones puntuales. Segunda (Sección 2.6): de diez subgrupos examinados, el peor se verá mal por azar, y un hallazgo en un subgrupo debe confirmarse con datos nuevos antes de actuar en consecuencia.

La <strong>model card</strong> de Mitchell y colaboradores es un documento estructurado que indica, en campos fijos, qué es un modelo y para qué sirve, con qué datos se entrenó y evaluó, cómo se desempeña en general y por subgrupo, y cuáles son sus limitaciones y consideraciones éticas.<sup class="c"><a href="#r15">[15]</a></sup> El bloque arma una a partir de las cantidades calculadas más arriba, de modo que cada campo sea rastreable hasta un bloque de este notebook.



In [ ]:
# Paso 7b: una model card armada a partir de las cantidades calculadas (Mitchell et al. 2019).
r5 <- thr_rows[thr_rows$cost_ratio==5,]
card <- data.frame(field=c("Model details","Intended use","Out-of-scope use","Training data","Evaluation data",
  "Discrimination","Calibration","Operating point","Performance at operating point","Clinical utility",
  "Subgroup performance","Ethical considerations","Caveats and limitations"),
  content=c(
  sprintf("%s; standardise -> tuned estimator (%s) -> Platt (sigmoid) calibration on 5-fold training predictions. R %s, e1071/randomForest/nnet.", fam_label[chosen_fam], paste(names(best_overall$best),unlist(best_overall$best),sep="=",collapse=", "), getRversion()),
  "Probability that a fine-needle aspirate is malignant, from nine nuclear-morphology scores, to support (not replace) the decision to biopsy in a referral setting.",
  "Screening populations (prevalence far below 35%), images from other laboratories or feature software, any use without a pathologist in the loop.",
  sprintf("%d aspirates (Wisconsin, mlbench copy), %.1f%% malignant.", length(ytr), 100*mean(ytr=="mal")),
  sprintf("%d held-out aspirates from the same source (internal validation only), %.1f%% malignant. No external validation.", length(yte), 100*mean(yte=="mal")),
  sprintf("ROC-AUC %.3f on the test set; nested-CV AUC %.3f +/- %.3f.", auc_te, nested$outer_AUC[which.max(nested$outer_AUC)], nested$AUC_sd[which.max(nested$outer_AUC)]),
  sprintf("Brier score %.3f; reliability curve close to the diagonal (Figure 8).", brier),
  sprintf("Threshold %.2f, chosen on cross-validated training predictions for a cost ratio of 5.", T_STAR),
  sprintf("Sensitivity %.3f, specificity %.3f; %d false negatives and %d false positives among %d test cases.", r5$sensitivity, r5$specificity, r5$test_FN, r5$test_FP, length(yte)),
  "Net benefit above 'biopsy all' and 'biopsy none' for threshold probabilities 0.05-0.50 (Figure 9).",
  "Equal within sampling error across two random hospitals; no demographic variables, so fairness by age/ethnicity/socioeconomic group is UNTESTED.",
  "Label is the biopsy result; features depend on one laboratory's imaging. Any deployment requires local validation, monitoring and informed patients.",
  "Single-site, decades-old data; small test set (wide intervals); probabilities valid only at training prevalence; not evaluated prospectively."), stringsAsFactors=FALSE)
knitr::kable(card, col.names=c("Field","Content"))



Cada fila de la model card es un número calculado más arriba o una declaración de lo que <em>no</em> se hizo —solo validación interna, equidad no evaluada, prevalencia de uso restringida, sin evaluación prospectiva—, y el segundo tipo importa más: el lector sabe qué evidencia falta antes de decidir si usar el modelo.

**Estándares de reporte.** <strong>TRIPOD+AI</strong> (2024) rige el reporte del desarrollo y la validación de un modelo de predicción —datos, desenlace, tamaño muestral, construcción del modelo, cada medida de desempeño con su incertidumbre, equidad, disponibilidad del código y de los datos—: la model card convertida en lista de verificación para un manuscrito.<sup class="c"><a href="#r16">[16]</a></sup> <strong>DECIDE-AI</strong> (2022) rige la <em>evaluación clínica temprana</em> de un sistema de apoyo a la decisión y pide lo que un artículo de desarrollo no puede contener: cómo interactuaron los usuarios con él, con qué frecuencia lo anularon, qué errores surgieron, cómo se monitoreó la seguridad.<sup class="c"><a href="#r18">[18]</a></sup> <strong>CONSORT-AI</strong> y <strong>SPIRIT-AI</strong> (2020) extienden los estándares para ensayos aleatorizados y protocolos a las intervenciones con un componente de IA.<sup class="c"><a href="#r17">[17]</a></sup> Los tres responden las tres preguntas de la Sección 3.3: ¿predice el modelo?, ¿pueden usarlo los clínicos?, ¿mejora la atención el usarlo?

### 4.7 Por qué la mayoría de los modelos de predicción nunca llega a la práctica

Cada año se publican miles de modelos de predicción clínica y muy pocos se usan. El COVID-19 dio la demostración más clara: la revisión sistemática viva de Wynants y colaboradores evaluó cada modelo diagnóstico o pronóstico publicado —232 a mediados de 2020, varios cientos desde entonces— con la herramienta de riesgo de sesgo PROBAST. Casi todos tenían un riesgo de sesgo alto o poco claro, por razones que forman un catálogo de las advertencias de este curso —controles no representativos, exclusión de pacientes que aún no habían presentado el desenlace, sobreajuste por pocos eventos, reporte demasiado vago para permitir la validación, desempeño estimado con los datos de desarrollo—, y ninguno fue recomendado para uso clínico.<sup class="c"><a href="#r19">[19]</a></sup> La falla no estuvo en los algoritmos —en su mayoría regresiones logísticas y ensambles de árboles—, sino en el <em>proceso</em>: evaluación, fuga de datos, validación externa. La revisión de Topol llegó a la misma conclusión desde la dirección opuesta: de los muchos sistemas que igualan a los especialistas de manera retrospectiva, un puñado ha sido evaluado prospectivamente y menos aún en ensayos que midan desenlaces de los pacientes.<sup class="c"><a href="#r28">[28]</a></sup> El modelo del proyecto integrador, de un solo sitio y con validación interna, es un modelo en etapa de desarrollo, y su model card lo dice.

<div class="exercise"><h4>Ejercicio 4</h4><p>Fuerza el MLP como la familia seleccionada (<code>chosen_fam <- "mlp"</code> antes del paso 4) y compara la curva de calibración, la curva de decisión y el umbral de operación con los de la SVM. Luego introduce una fuga deliberada —estandariza sobre los 683 casos antes de la división— y observa si cambia algún número de la model card. ¿Por qué esta fuga en particular es pequeña aquí, y cuándo sería grande?</p></div>

## 5. Ética profesional en el trabajo con datos (9.4)

La Unidad 1 introdujo la ética de los datos de salud a través de la reidentificación, el dataset shift, el sesgo en las etiquetas y el marco regulatorio. Esta sección se construye sobre ella en seis partes: el ciclo de vida de los datos; la equidad definida con la precisión suficiente para poder medirla; la explicabilidad y sus límites; la privacidad más allá de la desidentificación; la gobernanza y la ley; la lista de verificación final. El registro es el de Char, Shah y Magnus: las preguntas éticas no están separadas de las de ingeniería, porque las elecciones de etiqueta, datos, umbral y despliegue <em>son</em> las elecciones éticas.<sup class="c"><a href="#r32">[32]</a></sup>

### 5.1 El ciclo de vida de los datos y los deberes del ingeniero

Los datos sobre pacientes pasan por una secuencia de estados, y el ingeniero que construye un sistema que aprende es responsable en cada uno de ellos.



In [ ]:
# El ciclo de vida de los datos con el deber del ingeniero en cada etapa (esquema).
stg <- list(c("Collection","lawful basis; minimisation;\nfit for the stated purpose"),
  c("Consent","informed, specific, revocable;\nethics-committee approval"),
  c("Storage","encryption; pseudonymisation;\nkey kept separately"),
  c("Access","role-based; logged;\nneed-to-know; data-use agreement"),
  c("Analysis","S1 methodology; fairness audit;\nno re-identification attempts"),
  c("Publication","reporting standards; model card;\nno individual-level disclosure"),
  c("Retention /\ndeletion","retain only as long as justified;\nsecure deletion; right to erasure"))
plot.new(); plot.window(xlim=c(0,10), ylim=c(0,4.7)); w <- 9.4/length(stg)
for (i in seq_along(stg)) { x <- 0.3 + (i-1)*w
  rect(x, 2.6, x+w*0.86, 3.5, col=adjustcolor(BLUE,0.15), border=BLUE)
  text(x+w*0.43, 3.05, stg[[i]][1], font=2, col=BLUE, cex=0.72)
  yd <- ifelse(i%%2==1, 1.85, 1.0); segments(x+w*0.43, 2.6, x+w*0.43, yd+0.28, col=GREY, lty=3)
  text(x+w*0.43, yd, stg[[i]][2], cex=0.6)
  if (i < length(stg)) arrows(x+w*0.86, 3.05, x+w, 3.05, length=0.06, lwd=1) }
text(5, 0.25, "the cycle closes: data retained for a new purpose require a new basis and, usually, new consent", col=GREY, font=3, cex=0.7)
title("The data life cycle and the engineer's duty at each stage", cex.main=0.95)



<p class="caption"><b>Figura 10.</b> Siete etapas de la vida de un conjunto de datos de salud con el deber del ingeniero en cada una —habitualmente la misma persona que solicitó la extracción, guarda la clave de los seudónimos y decide qué se publica—.</p>

**Recolección y propósito.** Solo para un propósito declarado y solo en la medida necesaria (<em>minimización</em>); el GDPR y la Ley 21.719 de Chile hacen ilícito "recolectar todo" sin una base legal, y los datos recolectados para la atención no quedan por ello disponibles para la investigación. **Consentimiento y revisión ética.** Consentimiento informado, específico y revocable, o una dispensa justificada, más la aprobación de un comité de ética de la investigación (en Chile, el <em>Comité Ético Científico</em>, Ley 20.120). **Almacenamiento y acceso.** Seudonimización con la clave guardada por separado, cifrado, acceso basado en roles y registrado, un acuerdo de uso de datos —el ingeniero que copia una extracción a un computador portátil ha incumplido los cuatro—. **Análisis.** La metodología de S1 es un requisito ético —un modelo sobrestimado por fuga de datos daña a los pacientes en los que falla—, con una auditoría de equidad y sin intentos de reidentificación no autorizados. **Publicación.** Los estándares de la Sección 4.6; ningún individuo expuesto; código y, cuando sea lícito, datos compartidos. **Retención y eliminación.** Solo mientras lo requiera el propósito, con eliminación segura y respetando el derecho de supresión —y un modelo entrenado puede retener por sí mismo información sobre individuos (Sección 5.4)—.

### 5.2 Equidad: definiciones, el resultado de imposibilidad y una auditoría

"Equitativo" (fair) tiene varios significados precisos para un clasificador, y entran en conflicto. Con el desenlace \(Y\), la decisión \(\hat Y\), la probabilidad predicha \(\hat p\) y un atributo sensible \(A\) con grupos \(a, b\):

<div class="eq">\[\begin{aligned}
\text{Demographic parity:}\quad & P(\hat Y = 1 \mid A = a) = P(\hat Y = 1 \mid A = b) \\
\text{Equalised odds:}\quad & P(\hat Y = 1 \mid Y = y, A = a) = P(\hat Y = 1 \mid Y = y, A = b) \quad \text{for } y \in \{0, 1\} \\
\text{Calibration within groups:}\quad & P(Y = 1 \mid \hat p = p, A = a) = P(Y = 1 \mid \hat p = p, A = b) = p \quad \text{for all } p
\end{aligned}\]
<span class="where">La misma fracción de cada grupo marcada / igual sensibilidad e igual tasa de falsos positivos / un 0.3 predicho significa 30 % en todos los grupos.</span></div>

La paridad demográfica (demographic parity) es apropiada cuando las tasas base <em>deberían</em> ser iguales y cualquier diferencia es sesgo en las etiquetas —el caso de Obermeyer de la Unidad 1—.<sup class="c"><a href="#r29">[29]</a></sup> Es inapropiada cuando las tasas base difieren genuinamente: la incidencia del cáncer de mama aumenta con la edad, y marcar fracciones iguales de personas de 30 y de 65 años dejaría sin detectar cánceres en las mayores o sobretrataría a las jóvenes. Las equalised odds y la calibración son entonces los criterios —y Chouldechova (analizando el puntaje de reincidencia COMPAS) y Kleinberg, Mullainathan y Raghavan (en un teorema general) demostraron que <strong>cuando las tasas base difieren entre grupos y el clasificador es imperfecto, la calibración dentro de los grupos y la igualdad de las tasas de falsos positivos y de falsos negativos no pueden cumplirse todas a la vez</strong>—.<sup class="c"><a href="#r20">[20]</a>,<a href="#r21">[21]</a></sup> Hay que <em>elegir</em> qué equidad satisfacer, y la elección es un juicio de valor sobre los daños de la aplicación. El bloque audita el modelo del proyecto integrador según un grupo de edad sintético en el que los pacientes mayores son malignos con más frecuencia, como ocurre en la realidad.



In [ ]:
# Auditoría de equidad según un grupo de edad sintético cuya tasa base de malignidad difiere (pacientes mayores: más malignos).
set.seed(1); older <- ifelse(yte=="mal", runif(length(yte))<0.70, runif(length(yte))<0.35)   # P(mayor|maligno)=.70, P(mayor|benigno)=.35
age_group <- ifelse(older, ">= 55 years", "< 55 years")
au <- subgroup_report(yte, p_te, age_group, T_STAR)
au$false_positive_rate <- 1 - au$specificity
au$mean_predicted_p <- sapply(au$group, function(gg) mean(p_te[age_group==gg]))
au$observed_rate <- au$prevalence
au <- au[, c("group","n","prevalence","flagged","sensitivity","false_positive_rate","mean_predicted_p","observed_rate","AUC")]
cat("demographic parity -> 'flagged'; equalised odds -> sensitivity & FPR; calibration -> mean predicted vs observed\n")
num <- setdiff(names(au), c("group","n")); au[,num] <- round(au[,num],3); print(au, row.names=FALSE)



La <em>paridad demográfica</em> falla por construcción —el grupo de mayor edad se marca con mucha más frecuencia (alrededor de 0.51 frente a 0.22) porque tiene mucha más malignidad—, y ese es el comportamiento correcto. Las <em>equalised odds</em> se cumplen aproximadamente (sensibilidad similar, ambas en el techo, y tasas de falsos positivos dentro del error de muestreo de grupos de este tamaño), y también la <em>calibración dentro de los grupos</em> (la probabilidad predicha media sigue a la malignidad observada: alrededor de 0.49 frente a un 0.49 observado en el grupo de mayor edad, y 0.20 frente a 0.21 en el más joven). La auditoría aprueba en los criterios que importan aquí y falla en el que no importa, y la model card debe decir cuál es cuál. Si la edad hubiera estado correlacionada también con las <em>características</em> —pacientes mayores en un escáner más antiguo—, el modelo podría haberla aprendido como un atajo y las tasas de error habrían divergido.

<div class="box warn"><h4>La equidad no es una propiedad solo del modelo</h4><p>Un modelo equitativo en su conjunto de prueba puede producir resultados inequitativos al desplegarse —si un grupo tiene menos acceso a la consulta, o si los clínicos lo anulan de manera distinta según el paciente—. La equidad se audita a nivel del sistema y de sus resultados a lo largo del tiempo (Sección 5.6). A la inversa, "no usamos la etnia como característica" no garantiza nada: la Unidad 1 mostró que el sesgo entra solo a través de la etiqueta.</p></div>

### 5.3 Explicabilidad y transparencia

Un clínico al que se le pide actuar según la salida de un modelo necesita saber <em>por qué</em> —para contrastarla con el paciente, para detectar fallas, para asumir la responsabilidad que el modelo no puede asumir—. Las explicaciones son **globales** (de qué depende el modelo), **locales** (por qué un paciente recibió una salida determinada) o **intrínsecas** —regresión logística, un árbol poco profundo, un sistema de puntaje, cuyo cálculo es la explicación—. El bloque entrega dos explicaciones globales, implementadas en R base: la <strong>importancia por permutación</strong> (permutation importance), la pérdida de AUC de prueba cuando se baraja una característica, y la <strong>dependencia parcial</strong> (partial dependence), la predicción promedio a medida que varía una característica.



In [ ]:
# Explicaciones globales: importancia por permutación (caída del AUC de prueba al barajar una característica) y dependencia parcial.
sc <- std_fit(Xtr)
svm_final <- svm(std_ap(sc,Xtr), ytr, kernel="radial", cost=best_overall$best$C, gamma=best_overall$best$gamma, probability=TRUE)
predict_final <- function(Xn){ raw <- attr(predict(svm_final, std_ap(sc,Xn), probability=TRUE),"probabilities")[,"mal"]
  plogis(co[1] + co[2]*qlogis(clamp(raw))) }
set.seed(0); base_auc <- auc_of(yte, predict_final(Xte))
imp <- sort(sapply(colnames(Xte), function(f) mean(replicate(10, { Xp<-Xte; Xp[,f]<-sample(Xp[,f]); base_auc - auc_of(yte, predict_final(Xp)) }))), decreasing=TRUE)
cat("permutation importance (mean drop in test AUC over 10 shuffles), top 6:\n"); print(round(head(imp,6),4))
cat(sprintf("features whose shuffling changes the AUC by < 0.001: %d of %d\n", sum(abs(imp)<0.001), length(imp)))
top2 <- names(imp)[1:2]
par(mfrow=c(1,3), mar=c(4,7,2,1))
barplot(rev(head(imp,9)), horiz=TRUE, las=1, col=BLUE, cex.names=0.7, main="Permutation importance", xlab="drop in test AUC", cex.main=0.85)
par(mar=c(4,4,2,1))
for (f in top2) { gx <- seq(min(Xte[,f]), max(Xte[,f]), length.out=20)
  pd <- sapply(gx, function(v){ Xp<-Xte; Xp[,f]<-v; mean(predict_final(Xp)) })
  plot(gx, pd, type="l", col=ORANGE, lwd=2, xlab=f, ylab="mean predicted p(malignant)", main=paste("partial dependence:", f), cex.main=0.8) }



<p class="caption"><b>Figura 11.</b> Izquierda: importancia por permutación de las características más influyentes —la caída media del AUC de prueba cuando se baraja la columna—. "Bare nuclei" (núcleos desnudos) y "clump thickness" (grosor del agrupamiento) encabezan la lista; todas las caídas son inferiores a 0.01 porque el AUC está cerca de su techo y las características son redundantes, y 7 de 9 pueden barajarse prácticamente sin efecto. Derecha: dependencia parcial respecto de las dos características principales —ambas aumentan con el puntaje, como predice la patología—; el rango es estrecho porque la curva promedia sobre las demás características.</p>

Ambas herramientas fallan con características correlacionadas —las variables correlacionadas con una característica barajada quedan intactas, de modo que una característica duplicada parece poco importante; la dependencia parcial evalúa el modelo en combinaciones que nunca ocurren— y ambas explican el <em>modelo</em>, no la <em>enfermedad</em>.

**Una explicación local y la alternativa interpretable.** Para un modelo lineal, la salida de un paciente se descompone de manera exacta: el logit es \(\beta_0 + \sum_j \beta_j z_j\), así que cada característica estandarizada contribuye con \(\beta_j z_j\). Para una caja negra, LIME y SHAP ajustan un sustituto local o calculan atribuciones basadas en teoría de juegos —aproximaciones—. Rudin sostiene que en las decisiones de alto riesgo la aproximación es el problema: puede equivocarse justo donde importa, y con datos tabulares un modelo interpretable suele desempeñarse igual de bien.<sup class="c"><a href="#r22">[22]</a></sup> El bloque pone a prueba esta afirmación: ajusta la regresión logística de la Unidad 2, compara su AUC con el de la caja negra y explica de manera exacta a un paciente limítrofe.



In [ ]:
# Un modelo interpretable junto a la caja negra, y una explicación local exacta de un paciente de prueba limítrofe.
Ztr <- std_ap(sc,Xtr); Zte <- std_ap(sc,Xte)
logit_model <- glm(y ~ ., data=data.frame(Ztr, y=as.integer(ytr=="mal")), family=binomial)
p_lr <- predict(logit_model, data.frame(Zte), type="response")
cat(sprintf("logistic regression: test AUC %.3f, Brier %.3f  |  black box: AUC %.3f, Brier %.3f\n",
    auc_of(yte,p_lr), mean((p_lr-(yte=="mal"))^2), auc_te, brier))
i <- which.min(abs(p_te-0.5)); z <- Zte[i,]; b <- coef(logit_model)
contrib <- sort(b[-1]*z)
cat(sprintf("patient %d: true class %s; black-box p = %.2f; logistic p = %.2f\n", i, as.character(yte[i]), p_te[i], p_lr[i]))
cat(sprintf("log-odds = intercept %.2f + sum of contributions %.2f = %.2f\n", b[1], sum(b[-1]*z), b[1]+sum(b[-1]*z)))
shown <- contrib                                                            # las nueve características (son solo nueve, así que se muestran todas)
par(mar=c(4,9,2,1)); barplot(shown, horiz=TRUE, las=1, col=ifelse(shown<0,BLUE,ORANGE), cex.names=0.7,
  xlab="contribution to the log-odds of malignancy", main=sprintf("Why the logistic model gives patient %d p = %.2f", i, p_lr[i]), cex.main=0.8)
abline(v=0)



<p class="caption"><b>Figura 12.</b> Explicación local exacta de un paciente de prueba según la regresión logística: cada barra es un coeficiente multiplicado por el valor estandarizado del paciente, es decir, su contribución a las log-odds de malignidad. El paciente es aquel que la caja negra encontró más ambiguo (p cercana a 0.5): un grosor del agrupamiento ("clump thickness") alto empuja con fuerza hacia maligno, y puntajes bajos de núcleos desnudos y de nucléolos ("bare nuclei", "nucleoli") hacia benigno; la suma del intercepto y las contribuciones da el logit y la probabilidad logística impresa arriba.</p>

El AUC de prueba de la regresión logística está a menos de una milésima del de la caja negra —el argumento de Rudin en miniatura—: con nueve características tabulares bien diseñadas, el modelo interpretable no pierde nada y responde "¿por qué?" de manera exacta. La defensa de una caja negra debe basarse en una ganancia medible en la población de <em>despliegue</em>. Donde la ganancia existe —imágenes, señales, texto, donde las características se aprenden (Unidad 8)—, las explicaciones post hoc como los saliency maps son la única opción y son hipótesis sobre el modelo, no razones.

### 5.4 La privacidad revisitada: desidentificación, datos sintéticos, aprendizaje federado, privacidad diferencial

La Unidad 1 mostró que eliminar los identificadores no anonimiza. Tres técnicas adicionales responden preguntas distintas. **Datos sintéticos**: un modelo generativo entrenado con los registros reales muestrea registros nuevos con la misma distribución conjunta; un generador ajustado con pocos registros puede memorizarlos y, sin una garantía formal, su privacidad es una afirmación empírica que debe ponerse a prueba mediante ataques de inferencia de pertenencia (membership inference). El **aprendizaje federado** permite que varios hospitales entrenen un solo modelo sin compartir registros: en el <em>promediado federado</em> (federated averaging) de McMahan y colaboradores, cada sitio entrena localmente y envía solo parámetros a un servidor, que los promedia ponderándolos por el tamaño muestral y devuelve el promedio, ronda tras ronda.<sup class="c"><a href="#r23">[23]</a></sup> Rieke y colaboradores revisan su uso en salud digital y sus problemas abiertos: sitios con distribuciones distintas, calidad de datos desigual y parámetros que pueden, a su vez, filtrar información sobre los registros de entrenamiento —el aprendizaje federado es una herramienta de gobernanza, no una garantía de privacidad—.<sup class="c"><a href="#r24">[24]</a></sup> El bloque simula una ronda con tres "hospitales" obtenidos a partir del conjunto de entrenamiento.



In [ ]:
# Promediado federado, una ronda: tres hospitales ajustan localmente una regresión logística; solo viajan los coeficientes y estadísticas resumidas.
set.seed(0); site <- sample(0:2, length(ytr), replace=TRUE, prob=c(0.5,0.3,0.2))
ns <- sapply(0:2, function(s) sum(site==s))
means <- t(sapply(0:2, function(s) colMeans(Xtr[site==s,]))); vars <- t(sapply(0:2, function(s) apply(Xtr[site==s,],2,var)))
mu_g <- colSums(ns*means)/sum(ns)                                            # escalador global obtenido solo a partir de los resúmenes de cada sitio
sig_g <- sqrt(colSums(ns*(vars + (means - matrix(mu_g,3,ncol(Xtr),byrow=TRUE))^2))/sum(ns))
Zg <- function(Mx) scale(Mx, mu_g, sig_g)
local <- lapply(0:2, function(s) glm(y~., data=data.frame(Zg(Xtr[site==s,]), y=as.integer(ytr[site==s]=="mal")), family=binomial))
coef_fed <- as.numeric(sapply(local, coef) %*% ns / sum(ns))                 # el servidor promedia los coeficientes ponderándolos por n
Zte_g <- cbind(1, Zg(Xte)); p_fed <- plogis(Zte_g %*% coef_fed)
central <- glm(y~., data=data.frame(Zg(Xtr), y=as.integer(ytr=="mal")), family=binomial)
rows <- data.frame(model=c(sprintf("hospital %d alone (n = %d)", 1:3, ns), "federated average (1 round)", "centralised (all pooled)"),
  test_AUC=c(sapply(0:2, function(s) auc_of(yte, plogis(Zte_g %*% coef(local[[s+1]])))),
             auc_of(yte, as.numeric(p_fed)), auc_of(yte, predict(central, data.frame(Zg(Xte)), type="response"))))
rows$test_AUC <- round(rows$test_AUC,4); print(rows, row.names=FALSE)



El modelo federado, construido sin que ningún registro saliera de su hospital, iguala al centralizado con una diferencia de alrededor de una milésima de AUC y supera al modelo propio del hospital más pequeño. Bastó una ronda porque los sitios comparten una distribución y el modelo es convexo; las federaciones reales necesitan muchas rondas. Lo que sí viajó —nueve medias, nueve varianzas, diez coeficientes por sitio— plantea la pregunta que responde la privacidad diferencial.

La **privacidad diferencial** es la única de estas técnicas con una garantía matemática: un algoritmo aleatorizado \(\mathcal{M}\) es \(\varepsilon\)-diferencialmente privado si, para todo par de conjuntos de datos \(D, D'\) que difieren en un registro y todo conjunto de salidas \(S\), \(P(\mathcal{M}(D) \in S) \le e^{\varepsilon}\, P(\mathcal{M}(D') \in S)\) —la salida apenas cambia según esté o no una persona cualquiera en los datos—.<sup class="c"><a href="#r25">[25]</a></sup> El mecanismo básico agrega ruido calibrado según la <em>sensibilidad</em> de la consulta, lo máximo que un registro puede cambiarla: para un conteo, ruido de Laplace de escala \(1/\varepsilon\). El presupuesto \(\varepsilon\) se gasta con cada consulta y la garantía se compone. El bloque publica un conteo con tres presupuestos.



In [ ]:
# El mecanismo de Laplace: publicar '¿cuántos casos malignos hay en el conjunto de prueba?' con tres presupuestos de privacidad.
rlaplace <- function(n, b){ u <- runif(n)-0.5; -b*sign(u)*log(1-2*abs(u)) }
true_count <- sum(yte=="mal"); set.seed(0)
plot.new(); plot.window(xlim=c(true_count-40, true_count+40), ylim=c(0,0.9)); axis(1); axis(2)
cols <- c(GREY, ORANGE, BLUE); labs <- character()
for (j in seq_along(c(0.1,1,10))) { eps <- c(0.1,1,10)[j]
  rel <- true_count + rlaplace(5000, 1/eps); h <- hist(rel, breaks=seq(true_count-200,true_count+200,1), plot=FALSE)
  keep <- h$mids >= true_count-40 & h$mids <= true_count+40
  lines(h$mids[keep], h$density[keep], col=cols[j], lwd=2)
  labs[j] <- sprintf("epsilon = %g: within +/-%.0f (95%%)", eps, quantile(abs(rel-true_count),0.95)) }
abline(v=true_count, lty=2); legend("topleft", labs, col=cols, lwd=2, bty="n", cex=0.75)
title(sprintf("Differential privacy on a count (true = %d): smaller epsilon, stronger guarantee, noisier answer", true_count), cex.main=0.8)
mtext("released count", 1, line=2.2, cex=0.8)



<p class="caption"><b>Figura 13.</b> El conteo publicado por el mecanismo de Laplace con tres presupuestos. Con \(\varepsilon = 10\) el valor casi siempre queda a una o dos unidades de la verdad y la garantía es débil; con \(\varepsilon = 0.1\) la respuesta es inútil para un conjunto de datos de este tamaño (se desvía en decenas), pero no se puede inferir la presencia de ningún individuo. El compromiso entre privacidad y utilidad se convierte en un solo número que puede presupuestarse y auditarse.</p>

La privacidad diferencial se aplica a estadísticas publicadas (el censo de los Estados Unidos desde 2020), al entrenamiento de modelos (ruido en las actualizaciones del gradiente) y a los generadores de datos sintéticos. Su costo es la exactitud, especialmente para grupos pequeños —justo donde las auditorías de equidad más necesitan precisión—; esa tensión es un problema abierto.

### 5.5 Gobernanza y regulación

Cuatro cuerpos de normas enmarcan el trabajo; la Unidad 1 introdujo los dos primeros.

**Protección de datos.** En Chile, la Ley 19.628 (1999) sobre protección de la vida privada; la Ley 20.584 (2012) sobre derechos de los pacientes, que establece la confidencialidad de la ficha clínica; y la Ley 21.719, publicada en diciembre de 2024 y vigente desde el 1 de diciembre de 2026, que alinea el régimen con el GDPR, crea la Agencia de Protección de Datos Personales con facultades de fiscalización y sanción, y clasifica los datos de salud y genéticos como sensibles (ver Recursos en línea). Los estudiantes que se titulen en 2027 ejercerán bajo la nueva ley desde su primer día.

**Regulación de dispositivos médicos.** El software que informa una decisión clínica es un dispositivo médico: regulado por la FDA en los Estados Unidos; en la Unión Europea, por el Reglamento de Dispositivos Médicos (2017/745) y la <strong>Ley de Inteligencia Artificial</strong> (AI Act, Reglamento 2024/1689), que clasifica la IA en dispositivos médicos como de <em>alto riesgo</em>, con obligaciones de gestión de riesgos, gobernanza de datos, documentación, transparencia, supervisión humana, exactitud, robustez y ciberseguridad, implementadas gradualmente en 2025–2027 (ver Recursos en línea); en Chile, por el Instituto de Salud Pública (ISP) bajo el Código Sanitario y el Decreto 825/1998, con una política nacional de IA y un proyecto de ley de IA que sigue la estructura europea basada en el riesgo. Un modelo desplegado sin registro sanitario es un dispositivo médico no registrado, sea cual sea su AUC.

**Orientación ética.** La guía de la OMS de 2021 establece seis principios —autonomía; bienestar, seguridad e interés público; transparencia, explicabilidad e inteligibilidad; responsabilidad y rendición de cuentas; inclusividad y equidad; IA receptiva y sostenible— (ver Recursos en línea). Cada uno se concreta aquí: model card y estándares de reporte (transparencia), curva de decisión y umbral basado en costos (seguridad), auditoría de equidad (equidad), privacidad diferencial y consentimiento (autonomía), la lista de verificación (rendición de cuentas).

**Integridad en la investigación y ciencia abierta.** Los <strong>principios FAIR</strong> piden que los datos y metadatos sean localizables, accesibles, interoperables y reutilizables (Findable, Accessible, Interoperable, Reusable) —identificadores persistentes, metadatos ricos, acceso con condiciones claras, vocabularios estándar—, bajo controles de acceso cuando son sensibles.<sup class="c"><a href="#r26">[26]</a></sup> Las diez reglas de Sandve y colaboradores para la investigación computacional reproducible —registrar cómo se produjo cada resultado, archivar las versiones de los programas, registrar las semillas, conservar los datos crudos detrás de los gráficos, dar acceso público a los scripts— son los principios de diseño de este notebook: cada número es reproducido por un bloque de código con una semilla fija, y el notebook es público.<sup class="c"><a href="#r27">[27]</a></sup> La autoría sigue la misma lógica: quien contribuyó de manera sustancial y puede asumir la responsabilidad es autor. Un modelo entrenado por un estudiante con datos de un hospital y publicado sin conocimiento del hospital viola a la vez el acuerdo de uso de datos, la aprobación ética y las normas de autoría.

### 5.6 Una lista de verificación profesional para el bioingeniero que despliega un sistema que aprende

El curso cierra con dieciséis preguntas que hay que responder antes, durante y después de desplegar cualquier sistema que aprenda de datos sobre personas.

<div class="box key"><h4>Antes de construir</h4>
<ol>
<li><strong>Propósito.</strong> ¿Qué decisión informará el sistema, quién la toma y qué le ocurre a un paciente cuando se equivoca en cada dirección?</li>
<li><strong>Etiqueta.</strong> ¿Mide la etiqueta lo que pretendemos, o un sustituto (proxy) que codifica una inequidad? ¿Quién la asignó y con qué tasa de error?</li>
<li><strong>Base legal y consentimiento.</strong> ¿Bajo qué base legal y qué aprobación ética se usan los datos, y está este proyecto dentro de ese propósito?</li>
<li><strong>Representatividad.</strong> ¿De qué sitios, escáneres, años y poblaciones provienen los datos; qué grupos están ausentes?</li>
</ol></div>

<div class="box key"><h4>Durante la construcción</h4>
<ol start="5">
<li><strong>Fuga de datos.</strong> ¿Está bloqueado el conjunto de prueba, está cada paso de preprocesamiento dentro de la validación cruzada, se eligen los hiperparámetros y la familia en bucles anidados?</li>
<li><strong>Calibración y umbral.</strong> ¿Son las salidas probabilidades calibradas, a qué prevalencia, y se eligió el umbral a partir de los costos clínicos junto con los clínicos?</li>
<li><strong>Utilidad.</strong> ¿Muestra una curva de decisión un beneficio neto sobre las estrategias por defecto en todo el rango de umbrales plausibles?</li>
<li><strong>Equidad.</strong> ¿Qué criterio es apropiado y por qué; lo satisface el modelo en los subgrupos que importan, con intervalos de confianza?</li>
<li><strong>Explicación.</strong> ¿Podría un modelo interpretable hacer el trabajo? Si no, ¿qué explicaciones recibirán los usuarios y cómo se validaron?</li>
<li><strong>Privacidad.</strong> Datos seudonimizados, acceso registrado, modelo verificado en cuanto a memorización; ¿se justifica un entrenamiento federado o con privacidad diferencial?</li>
</ol></div>

<div class="box key"><h4>Antes y después del despliegue</h4>
<ol start="11">
<li><strong>Validación externa.</strong> ¿Se ha evaluado el modelo con datos del sitio y del período de despliegue?</li>
<li><strong>Documentación.</strong> ¿Model card, etiqueta model facts y un reporte que siga TRIPOD+AI, DECIDE-AI o CONSORT-AI?</li>
<li><strong>Regulación.</strong> ¿Es un dispositivo médico, registrado ante el ISP (o la FDA / un organismo notificado de la UE), que cumple las obligaciones de alto riesgo de la Ley de IA cuando corresponde?</li>
<li><strong>Supervisión humana y los five rights.</strong> ¿Quién puede anularlo, cómo se registra eso, y llega a la persona correcta en el formato correcto en el momento correcto?</li>
<li><strong>Monitoreo y retiro.</strong> ¿Quién monitorea el desempeño, la calibración y las tasas de error por subgrupo; qué gatilla el reentrenamiento o el retiro; quién rinde cuentas?</li>
<li><strong>Reproducibilidad.</strong> ¿Están archivados el código, las semillas, el entorno y, cuando sea lícito, los datos, de modo que cada número pueda regenerarse?</li>
</ol></div>

Un sistema para el cual no se pueden responder estas preguntas no está listo, sea cual sea su AUC. Los métodos de las Unidades 2–8 hacen posibles las respuestas; esta unidad trata de la obligación de darlas.

## 6. Un panorama de las aplicaciones actuales, para el Taller 6

Para el Taller 6, quince sistemas o estudios, cada uno con una referencia principal y su <em>nivel de evidencia</em> (Sección 3.3). Elige una fila, lee el artículo con la lista de verificación de la Sección 5.6 e informa cuáles de sus preguntas responde el artículo.

| # | Dominio | Sistema o estudio | Qué hace (método de este curso) | Nivel de evidencia | Ref. |
|---|---|---|---|---|---|
| 1 | Imágenes — oftalmología | IDx-DR, tamizaje autónomo de retinopatía diabética | Clasificador CNN sobre fotografías de fondo de ojo (U8) | Ensayo pivotal prospectivo en atención primaria; autorización de la FDA para uso autónomo, 2018 | <sup class="c"><a href="#r28">[28]</a></sup> |
| 2 | Imágenes — radiología, patología, dermatología | Clasificadores de aprendizaje profundo comparados con especialistas | CNN, transfer learning (U8) | Metaanálisis de 82 estudios: equivalencia en promedio; 25 con validación externa; alto riesgo de sesgo | <sup class="c"><a href="#r10">[10]</a></sup> |
| 3 | Imágenes — dermatología | Clasificación de lesiones cutáneas a nivel de dermatólogo | CNN con fine-tuning a partir de ImageNet | Estudio retrospectivo de lectores frente a dermatólogos sobre lesiones confirmadas por biopsia | <sup class="c"><a href="#r28">[28]</a></sup> |
| 4 | Imágenes — todas las especialidades | Calidad del diseño de los estudios de IA frente a clínicos | — | Revisión sistemática: 81 estudios no aleatorizados, 2 ensayos; resúmenes que exageran | <sup class="c"><a href="#r11">[11]</a></sup> |
| 5 | Señales — cardiología, wearables | ECG con IA y clasificación del ritmo con relojes inteligentes | CNN sobre ECG y fotopletismografía (U6/U8) | Grandes validaciones retrospectivas; wearables con autorización de la FDA; evaluación prospectiva de las notificaciones de relojes inteligentes | <sup class="c"><a href="#r28">[28]</a></sup> |
| 6 | Predicción con registros clínicos electrónicos (EHR) | Modelo propietario de sepsis desplegado en cientos de hospitales de EE. UU. | Regresión logística penalizada sobre variables del registro (U2) | Validación externa tras el despliegue: sensibilidad 33 %, VPP 12 % — muy por debajo de lo declarado en el desarrollo | <sup class="c"><a href="#r33">[33]</a></sup> |
| 7 | Predicción con EHR | Sepsis Watch (Duke University Hospital) con una etiqueta model facts | Modelo profundo sobre datos del registro en flujo continuo, con una etiqueta model facts de una página | Desplegado con monitoreo prospectivo; el ejemplo de referencia de documentación orientada al usuario | <sup class="c"><a href="#r31">[31]</a></sup> |
| 8 | Predicción con EHR | Predicción de deterioro, reingreso y mortalidad | Regresión, ensambles de árboles, redes profundas (U2, U4, U6) | Revisión: sólidos resultados retrospectivos; las barreras son los flujos de datos, el flujo de trabajo y la evaluación de desenlaces | <sup class="c"><a href="#r30">[30]</a></sup> |
| 9 | Predicción con EHR | Modelos diagnósticos y pronósticos de COVID-19 | Mayormente regresión logística y ensambles de árboles (U2, U4) | Revisión viva: cientos de modelos, casi todos con alto riesgo de sesgo, ninguno recomendado | <sup class="c"><a href="#r19">[19]</a></sup> |
| 10 | Salud poblacional | Algoritmo comercial que asigna recursos de gestión de cuidados | Regresión que predice el costo como sustituto de la necesidad (U2) | Desplegado a gran escala; una auditoría encontró un gran sesgo racial a través de la etiqueta | <sup class="c"><a href="#r29">[29]</a></sup> |
| 11 | Texto clínico | Extracción de información de notas clínicas | Pipelines basados en reglas y en bolsa de palabras (Sección 2.8) | Revisión de 263 estudios; mayormente de una sola institución, rara vez con validación externa | <sup class="c"><a href="#r7">[7]</a></sup> |
| 12 | Imágenes multicéntricas | Consorcios de aprendizaje federado para imágenes médicas | Promediado federado entre hospitales (Sección 5.4) | Revisión de los primeros despliegues; problemas abiertos de heterogeneidad y privacidad | <sup class="c"><a href="#r24">[24]</a></sup> |
| 13 | Biología estructural | Predicción de estructura de proteínas con AlphaFold2 | Red de atención sobre secuencias y alineamientos (BGM324) | Evaluación ciega (CASP14); exactitud cercana a la experimental; base de datos pública de estructuras | <sup class="c"><a href="#r34">[34]</a></sup> |
| 14 | Descubrimiento de fármacos | Clasificación QSAR de inhibidores de ureasa y de TRPV1 | Ensambles de árboles sobre descriptores; reposicionamiento; docking y simulación (U4; BGM324) | Estudios computacionales con validación cruzada; candidatos priorizados para ensayo | <sup class="c"><a href="#r8">[8]</a>,<a href="#r9">[9]</a></sup> |
| 15 | Modelos de lenguaje | Med-PaLM: gran modelo de lenguaje para responder preguntas médicas | Modelo de lenguaje Transformer (U1) | Evaluación con benchmarks y calificación de médicos; sin evaluación clínica | <sup class="c"><a href="#r35">[35]</a></sup> |

Tres patrones: el nivel de evidencia es independiente de la sofisticación del método —el modelo más simple (fila 6) es el más ampliamente desplegado y el que se demostró que falla—; los sistemas con evidencia prospectiva (filas 1, 5, 7) son acotados y se evaluaron a nivel de la decisión; y las auditorías y revisiones (filas 4, 6, 9, 10) son las más instructivas, porque muestran qué preguntas de la lista de verificación no se plantearon.

## 7. Resumen del curso

Ahora deberías ser capaz de explicar, sin apuntes:

- KDD y CRISP-DM; las cuatro familias de tareas; por qué la mayor parte del trabajo ocurre antes y después del algoritmo.
- Soporte, confianza, lift; la unión y la poda de Apriori; por qué muchas reglas producen reglas espurias y cómo una distribución nula por permutación y Bonferroni las ponen en evidencia.
- Los isolation forests ("diferente" no es "enfermo"); la bolsa de palabras y lo que pierde.
- Las cinco tareas de imágenes; qué hace diferentes a las imágenes médicas; Dice e IoU; la base de evidencia; cómo un cambio de escáner quiebra un modelo validado honestamente.
- De MYCIN e INTERNIST-1 al apoyo a la decisión; los five rights; las razones de verosimilitud y el nomograma de Fagan; por qué la probabilidad de un clasificador está ligada a la prevalencia de su entrenamiento.
- El modelo diagnóstico honesto: validación cruzada anidada, calibración, curva de decisión, umbral basado en costos, verificación por subgrupos, model card; los tres estándares de reporte; por qué la mayoría de los modelos nunca llega a la práctica.
- El ciclo de vida de los datos; los tres criterios de equidad y el resultado de imposibilidad; la explicación y sus límites; los datos sintéticos, el promediado federado, la privacidad diferencial; el marco legal, los principios de la OMS, FAIR, la reproducibilidad; la lista de verificación de dieciséis preguntas.

## 8. Autoevaluación

<details><summary>1. ¿Qué parte del curso sirve a cada adjetivo de KDD —válido, novedoso, útil, comprensible— y cuál de ellos no cumple por defecto una caja negra con AUC 0.99?</summary>Válido: Suplemento S1. Novedoso: conocimiento del dominio (Sección 2.1). Útil: curvas de decisión y los five rights (Secciones 4.2, 4.4). Comprensible: Sección 5.3 —el que la caja negra no cumple por defecto—.</details>

<details><summary>2. De 200 pacientes, 80 tienen hipertensión, 60 toman una estatina y 36 ambas cosas. ¿Soporte, confianza y lift de "hipertensión ⇒ estatina"?</summary>Soporte 36/200 = 0.18; confianza 36/80 = 0.45; lift 0.45/0.30 = 1.5: una asociación positiva, cuya significancia requiere una prueba y cuya causalidad no puede conocerse a partir de estos datos.</details>

<details><summary>3. ¿Por qué se podó {HTN, DM, ACEi} sin contarlo, y qué cambia si no se poda?</summary>Su subconjunto {DM, ACEi} no era frecuente, así que por la propiedad Apriori no puede ser frecuente. Sin poda el resultado no cambia, pero se cuentan muchos más candidatos por nivel —y en una base de datos real el número de candidatos es lo que vuelve intratable el problema—.</details>

<details><summary>4. Un colega mina 10,000 pares fármaco–evento e informa cada par con lift superior a 2. ¿Qué está mal?</summary>Con 10,000 pruebas, muchos pares superan cualquier lift fijo por azar, especialmente los poco frecuentes (Figura 3). Hay que poner a prueba cada par, corregir por el número de pruebas (Bonferroni o tasa de falsos descubrimientos), tratar con cautela los pares poco frecuentes y confirmar los sobrevivientes en datos independientes y mediante revisión clínica.</details>

<details><summary>5. |S| = 500 píxeles, |G| = 400, solapamiento 300. Calcula Dice e IoU y verifica la identidad.</summary>Dice = 600/900 = 0.667; IoU = 300/600 = 0.500; 2 × 0.5/1.5 = 0.667. ✓</details>

<details><summary>6. En la Figura 6 la exactitud colapsó con un "escáner distinto", aunque las exactitudes de validación cruzada y de prueba coincidían. ¿Por qué ninguna advirtió el problema?</summary>Ambas se calcularon sobre la distribución de entrenamiento y no dicen nada sobre otra. Solo la evaluación con datos del otro escáner, o el entrenamiento con aumento de datos que incluya la variación, aborda el cambio.</details>

<details><summary>7. Sensibilidad 0.95, especificidad 0.80: ¿LR⁺ y probabilidad post-test tras un resultado positivo con probabilidad pre-test de 0.10 y de 0.50?</summary>LR⁺ = 0.95/0.20 = 4.75. Odds 0.111 → 0.528 → p = 0.35. Odds 1 → 4.75 → p = 0.83. El mismo resultado da 35 % u 83 % según a quién se le hizo la prueba.</details>

<details><summary>8. El código eligió la familia con el AUC más alto en las particiones externas. ¿Es insesgado ese AUC? ¿Y el AUC del conjunto de prueba?</summary>El AUC externo seleccionado es levemente optimista —el máximo de tres estimaciones ruidosas—. El AUC del conjunto de prueba es insesgado para todo el procedimiento, incluida la selección, porque el conjunto de prueba no participó en ninguna elección.</details>

<details><summary>9. En la Figura 9, ¿qué significa un beneficio neto por encima de "biopsy everyone" con p_t = 0.20, y por qué todas las curvas se juntan cerca de p_t = 0?</summary>Con p_t = 0.20 un clínico acepta cuatro biopsias innecesarias por cada cáncer; una curva más alta significa que actuar según el modelo encuentra más cánceres, descontado el daño, a esa tasa de intercambio. Cerca de p_t = 0 un falso positivo no conlleva daño, así que biopsiar a todos es óptimo.</details>

<details><summary>10. El grupo de mayor edad fue marcado con mucha más frecuencia. ¿Es inequitativo? ¿Qué criterio falla, cuáles se cumplen, y por qué no los tres?</summary>La paridad demográfica falla, de manera apropiada, porque la malignidad es más común en el grupo de mayor edad; las equalised odds y la calibración dentro de los grupos se cumplen aproximadamente. Con tasas base distintas y un clasificador imperfecto, la calibración y la igualdad de las tasas de error no pueden cumplirse ambas exactamente, de modo que una elección justificada es inevitable.</details>

<details><summary>11. En el bloque federado, ¿qué salió de cada hospital, y por qué la privacidad diferencial sigue siendo relevante?</summary>El tamaño muestral, nueve medias, nueve varianzas y diez coeficientes —funciones de los datos de los pacientes que pueden filtrar información sobre individuos (la media de un sitio pequeño se desplaza con un solo paciente extremo; los coeficientes pueden memorizar registros poco frecuentes)—. Se necesita una garantía formal sobre las cantidades transmitidas para que la federación preserve la privacidad en lugar de ser simplemente local en cuanto a los datos.</details>

<details><summary>12. Un hospital quiere que el modelo del proyecto integrador se despliegue el próximo mes. ¿Qué cuatro preguntas de la lista de verificación responde este notebook de manera inadecuada?</summary>Validación externa (P11): solo validación interna con datos de un solo sitio. Equidad (P8): no hay variables demográficas. Regulación (P13): un dispositivo médico sin registro en el ISP. Supervisión y monitoreo (P14–15): no hay un plan para anulaciones, monitoreo ni retiro. La base legal para los datos locales (P3) es una quinta.</details>

## Glosario

<dl class="gloss">
<dt>Propiedad Apriori (Apriori property)</dt><dd>Todo subconjunto de un itemset frecuente es frecuente; la base del paso de poda.</dd>
<dt>Regla de asociación (association rule)</dt><dd>A ⇒ B entre itemsets disjuntos; coocurrencia, no causalidad.</dd>
<dt>Corrección de Bonferroni (Bonferroni correction)</dt><dd>Dividir el nivel de significancia por el número de pruebas realizadas.</dd>
<dt>Puntaje de Brier (Brier score)</dt><dd>Diferencia cuadrática media entre la probabilidad predicha y el desenlace 0/1; más bajo es mejor.</dd>
<dt>Calibración (calibration)</dt><dd>Concordancia entre las probabilidades predichas y las frecuencias observadas.</dd>
<dt>CDSS (clinical decision support system)</dt><dd>Sistema de apoyo a la decisión clínica.</dd>
<dt>Confianza / soporte / lift (confidence / support / lift)</dt><dd>P(B | A) / P(A ∪ B) / P(A, B) dividido por P(A)P(B), estimados a partir de las frecuencias de las transacciones; el lift es 1 bajo independencia.</dd>
<dt>CRISP-DM</dt><dd>Comprensión del negocio, comprensión de los datos, preparación de los datos, modelado, evaluación, despliegue.</dd>
<dt>Decision-curve analysis</dt><dd>Beneficio neto de un modelo a lo largo de las probabilidades umbral, frente a tratar a todos y a no tratar a nadie.</dd>
<dt>Paridad demográfica / equalised odds / calibración dentro de los grupos (demographic parity / equalised odds / calibration within groups)</dt><dd>Igual tasa de decisiones positivas / igual sensibilidad e igual tasa de falsos positivos / igual significado de una probabilidad predicha entre los grupos; incompatibles en general cuando las tasas base difieren.</dd>
<dt>Dice / IoU</dt><dd>2|S ∩ G|/(|S| + |G|) y |S ∩ G|/|S ∪ G|; Dice = 2 IoU/(1 + IoU).</dd>
<dt>Privacidad diferencial (differential privacy)</dt><dd>La distribución de las salidas cambia a lo más en un factor e^ε cuando se agrega o se elimina un registro.</dd>
<dt>Domain shift</dt><dd>Un cambio en la distribución de las entradas entre el entrenamiento y el despliegue.</dd>
<dt>Aprendizaje federado (federated learning)</dt><dd>Entrenamiento entre poseedores de datos mediante el intercambio de parámetros en lugar de registros.</dd>
<dt>Isolation forest</dt><dd>Detector de anomalías: los puntos aislados con pocas divisiones aleatorias son anómalos. No necesita etiquetas.</dd>
<dt>KDD (knowledge discovery in databases)</dt><dd>Descubrimiento de conocimiento en bases de datos; la minería de datos es uno de sus pasos.</dd>
<dt>Razón de verosimilitud (likelihood ratio)</dt><dd>LR⁺ = sensibilidad/(1 − especificidad); LR⁻ = (1 − sensibilidad)/especificidad; multiplica las odds pre-test.</dd>
<dt>Model card</dt><dd>Declaración estructurada de los detalles de un modelo, su uso previsto, sus datos, sus métricas por subgrupo y sus advertencias.</dd>
<dt>Validación cruzada anidada (nested cross-validation)</dt><dd>Bucle interno para los hiperparámetros dentro de cada partición de entrenamiento de un bucle externo para el desempeño.</dd>
<dt>Beneficio neto (net benefit)</dt><dd>TP/n − FP/n × p_t/(1 − p_t).</dd>
<dt>Dependencia parcial / importancia por permutación (partial dependence / permutation importance)</dt><dd>Predicción promedio a medida que varía una característica / pérdida de desempeño cuando se baraja una característica.</dd>
<dt>TRIPOD+AI, DECIDE-AI, CONSORT-AI</dt><dd>Estándares de reporte para el desarrollo y la validación de modelos, la evaluación clínica temprana y los ensayos aleatorizados con un componente de IA.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Fayyad U, Piatetsky-Shapiro G, Smyth P. From data mining to knowledge discovery in databases. <em>AI Magazine</em>. 1996;17(3):37–54. <a href="https://ojs.aaai.org/aimagazine/index.php/aimagazine/article/view/1230">ojs.aaai.org/aimagazine/…/view/1230</a> (página de la editorial; el DOI 10.1609/aimag.v17i3.1230 indicado por la editorial no está indexado en Crossref y no pudo verificarse al momento de escribir este texto)</li>
<li id="r2">Schröer C, Kruse F, Gómez JM. A systematic literature review on applying CRISP-DM process model. <em>Procedia Comput Sci</em>. 2021;181:526–534. <a href="https://doi.org/10.1016/j.procs.2021.01.199">doi:10.1016/j.procs.2021.01.199</a></li>
<li id="r3">Agrawal R, Imieliński T, Swami A. Mining association rules between sets of items in large databases. <em>Proc 1993 ACM SIGMOD Int Conf Management of Data</em>. 1993:207–216. <a href="https://doi.org/10.1145/170035.170072">doi:10.1145/170035.170072</a></li>
<li id="r4">Agrawal R, Srikant R. Fast algorithms for mining association rules in large databases. <em>Proc 20th Int Conf Very Large Data Bases (VLDB 1994)</em>. 1994:487–499. <a href="https://www.vldb.org/conf/1994/P487.PDF">vldb.org/conf/1994/P487.PDF</a> (artículo de congreso sin DOI)</li>
<li id="r5">Brin S, Motwani R, Silverstein C. Beyond market baskets: generalizing association rules to correlations. <em>Proc 1997 ACM SIGMOD Int Conf Management of Data</em>. 1997:265–276. <a href="https://doi.org/10.1145/253260.253327">doi:10.1145/253260.253327</a></li>
<li id="r6">Liu FT, Ting KM, Zhou Z-H. Isolation forest. <em>Proc 8th IEEE Int Conf Data Mining (ICDM 2008)</em>. 2008:413–422. <a href="https://doi.org/10.1109/ICDM.2008.17">doi:10.1109/ICDM.2008.17</a></li>
<li id="r7">Wang Y, Wang L, Rastegar-Mojarad M, et al. Clinical information extraction applications: a literature review. <em>J Biomed Inform</em>. 2018;77:34–49. <a href="https://doi.org/10.1016/j.jbi.2017.11.011">doi:10.1016/j.jbi.2017.11.011</a></li>
<li id="r8">Morales N, Valdés-Muñoz E, González J, et al. Machine learning-driven classification of urease inhibitors leveraging physicochemical properties as effective filter criteria. <em>Int J Mol Sci</em>. 2024;25(8):4303. <a href="https://doi.org/10.3390/ijms25084303">doi:10.3390/ijms25084303</a></li>
<li id="r9">González J, Palma JM, Benso B, et al. Integrative computational approaches for TRPV1 ion channel inhibitor discovery: an integrated machine learning, drug repurposing and molecular simulation approach. <em>J Chem Inf Model</em>. 2025;65(17):8957–8968. <a href="https://doi.org/10.1021/acs.jcim.5c00991">doi:10.1021/acs.jcim.5c00991</a></li>
<li id="r10">Liu X, Faes L, Kale AU, et al. A comparison of deep learning performance against health-care professionals in detecting diseases from medical imaging: a systematic review and meta-analysis. <em>Lancet Digit Health</em>. 2019;1(6):e271–e297. <a href="https://doi.org/10.1016/S2589-7500(19)30123-2">doi:10.1016/S2589-7500(19)30123-2</a></li>
<li id="r11">Nagendran M, Chen Y, Lovejoy CA, et al. Artificial intelligence versus clinicians: systematic review of design, reporting standards, and claims of deep learning studies. <em>BMJ</em>. 2020;368:m689. <a href="https://doi.org/10.1136/bmj.m689">doi:10.1136/bmj.m689</a></li>
<li id="r12">Miller RA, Pople HE, Myers JD. INTERNIST-1, an experimental computer-based diagnostic consultant for general internal medicine. <em>N Engl J Med</em>. 1982;307(8):468–476. <a href="https://doi.org/10.1056/NEJM198208193070803">doi:10.1056/NEJM198208193070803</a></li>
<li id="r13">Fagan TJ. Nomogram for Bayes's theorem. <em>N Engl J Med</em>. 1975;293(5):257. <a href="https://doi.org/10.1056/NEJM197507312930513">doi:10.1056/NEJM197507312930513</a></li>
<li id="r14">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r15">Mitchell M, Wu S, Zaldivar A, et al. Model cards for model reporting. <em>Proc Conf Fairness, Accountability, and Transparency (FAT* 2019)</em>. 2019:220–229. <a href="https://doi.org/10.1145/3287560.3287596">doi:10.1145/3287560.3287596</a></li>
<li id="r16">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r17">Liu X, Cruz Rivera S, Moher D, Calvert MJ, Denniston AK; SPIRIT-AI and CONSORT-AI Working Group. Reporting guidelines for clinical trial reports for interventions involving artificial intelligence: the CONSORT-AI extension. <em>Nat Med</em>. 2020;26(9):1364–1374. <a href="https://doi.org/10.1038/s41591-020-1034-x">doi:10.1038/s41591-020-1034-x</a></li>
<li id="r18">Vasey B, Nagendran M, Campbell B, et al. Reporting guideline for the early-stage clinical evaluation of decision support systems driven by artificial intelligence: DECIDE-AI. <em>Nat Med</em>. 2022;28(5):924–933. <a href="https://doi.org/10.1038/s41591-022-01772-9">doi:10.1038/s41591-022-01772-9</a></li>
<li id="r19">Wynants L, Van Calster B, Collins GS, et al. Prediction models for diagnosis and prognosis of covid-19: systematic review and critical appraisal. <em>BMJ</em>. 2020;369:m1328. <a href="https://doi.org/10.1136/bmj.m1328">doi:10.1136/bmj.m1328</a></li>
<li id="r20">Chouldechova A. Fair prediction with disparate impact: a study of bias in recidivism prediction instruments. <em>Big Data</em>. 2017;5(2):153–163. <a href="https://doi.org/10.1089/big.2016.0047">doi:10.1089/big.2016.0047</a></li>
<li id="r21">Kleinberg J, Mullainathan S, Raghavan M. Inherent trade-offs in the fair determination of risk scores. <em>Proc 8th Innovations in Theoretical Computer Science Conference (ITCS 2017)</em>, LIPIcs 67:43:1–43:23. Preprint de arXiv. <a href="https://doi.org/10.48550/arXiv.1609.05807">doi:10.48550/arXiv.1609.05807</a></li>
<li id="r22">Rudin C. Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. <em>Nat Mach Intell</em>. 2019;1(5):206–215. <a href="https://doi.org/10.1038/s42256-019-0048-x">doi:10.1038/s42256-019-0048-x</a></li>
<li id="r23">McMahan HB, Moore E, Ramage D, Hampson S, Agüera y Arcas B. Communication-efficient learning of deep networks from decentralized data. <em>Proc 20th Int Conf Artificial Intelligence and Statistics (AISTATS 2017), PMLR</em>. 2017;54:1273–1282. <a href="https://proceedings.mlr.press/v54/mcmahan17a.html">proceedings.mlr.press/v54/mcmahan17a.html</a> (las actas de PMLR no tienen DOI)</li>
<li id="r24">Rieke N, Hancox J, Li W, et al. The future of digital health with federated learning. <em>npj Digit Med</em>. 2020;3:119. <a href="https://doi.org/10.1038/s41746-020-00323-1">doi:10.1038/s41746-020-00323-1</a></li>
<li id="r25">Dwork C, Roth A. The algorithmic foundations of differential privacy. <em>Found Trends Theor Comput Sci</em>. 2014;9(3–4):211–407. <a href="https://doi.org/10.1561/0400000042">doi:10.1561/0400000042</a></li>
<li id="r26">Wilkinson MD, Dumontier M, Aalbersberg IJ, et al. The FAIR Guiding Principles for scientific data management and stewardship. <em>Sci Data</em>. 2016;3:160018. <a href="https://doi.org/10.1038/sdata.2016.18">doi:10.1038/sdata.2016.18</a></li>
<li id="r27">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r28">Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25(1):44–56. <a href="https://doi.org/10.1038/s41591-018-0300-7">doi:10.1038/s41591-018-0300-7</a></li>
<li id="r29">Obermeyer Z, Powers B, Vogeli C, Mullainathan S. Dissecting racial bias in an algorithm used to manage the health of populations. <em>Science</em>. 2019;366(6464):447–453. <a href="https://doi.org/10.1126/science.aax2342">doi:10.1126/science.aax2342</a></li>
<li id="r30">Rajkomar A, Dean J, Kohane I. Machine learning in medicine. <em>N Engl J Med</em>. 2019;380(14):1347–1358. <a href="https://doi.org/10.1056/NEJMra1814259">doi:10.1056/NEJMra1814259</a></li>
<li id="r31">Sendak MP, Gao M, Brajer N, Balu S. Presenting machine learning model information to clinical end users with model facts labels. <em>npj Digit Med</em>. 2020;3:41. <a href="https://doi.org/10.1038/s41746-020-0253-3">doi:10.1038/s41746-020-0253-3</a></li>
<li id="r32">Char DS, Shah NH, Magnus D. Implementing machine learning in health care — addressing ethical challenges. <em>N Engl J Med</em>. 2018;378(11):981–983. <a href="https://doi.org/10.1056/NEJMp1714229">doi:10.1056/NEJMp1714229</a></li>
<li id="r33">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065–1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r34">Jumper J, Evans R, Pritzel A, et al. Highly accurate protein structure prediction with AlphaFold. <em>Nature</em>. 2021;596(7873):583–589. <a href="https://doi.org/10.1038/s41586-021-03819-2">doi:10.1038/s41586-021-03819-2</a></li>
<li id="r35">Singhal K, Azizi S, Tu T, et al. Large language models encode clinical knowledge. <em>Nature</em>. 2023;620(7972):172–180. <a href="https://doi.org/10.1038/s41586-023-06291-2">doi:10.1038/s41586-023-06291-2</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R (incluye los datos BreastCancer de Wisconsin usados aquí). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group, TU Wien. Paquete de R (las implementaciones de SVM y de Bayes ingenuo). <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r39">Liaw A, Wiener M. Classification and regression by randomForest. <em>R News</em>. 2002;2(3):18–22. <a href="https://doi.org/10.32614/CRAN.package.randomForest">doi:10.32614/CRAN.package.randomForest</a></li>
<li id="r40">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r41">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002 (la referencia para <code>nnet</code>, <code>lda</code> y las funciones de modelado de R usadas a lo largo del notebook). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulo 14.2 para las reglas de asociación y el algoritmo Apriori; Capítulo 7 para la evaluación de modelos. PDF gratuito en el sitio de los autores.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em> (ISLR). 2nd ed. Springer; 2021. — Capítulo 5 para remuestreo y Capítulo 4 para los umbrales de clasificación; el compañero en R de todo el curso. PDF gratuito en statlearning.com.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Capítulo 1.5 para la teoría de decisión (costos, rechazo, la forma de Bayes en términos de odds).</li>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. — Capítulo 27 ("Philosophy, ethics and safety of AI") y Capítulo 12 para el razonamiento probabilístico.</li>
<li>Osheroff JA, Teich JM, Levick D, et al. <em>Improving Outcomes with Clinical Decision Support: An Implementer's Guide</em>. 2nd ed. HIMSS; 2012. — La fuente de los "five rights" de los CDSS.</li>
<li>Barocas S, Hardt M, Narayanan A. <em>Fairness and Machine Learning: Limitations and Opportunities</em>. MIT Press; 2023. — El texto de referencia para la Sección 5.2. Gratuito en línea en fairmlbook.org.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Task Views de CRAN "MachineLearning" y "Cluster" — <a href="https://cran.r-project.org/web/views/">cran.r-project.org/web/views</a>; los equivalentes en R de las secciones "Probability calibration", "Inspection" y "Novelty and outlier detection" de la guía de usuario de scikit-learn.</li>
<li>Declaración TRIPOD y lista de verificación TRIPOD+AI — <a href="https://www.tripod-statement.org">tripod-statement.org</a>.</li>
<li>EQUATOR Network (guías de reporte para la investigación en salud, incluidas CONSORT-AI, SPIRIT-AI y DECIDE-AI) — <a href="https://www.equator-network.org">equator-network.org</a>.</li>
<li>FDA, lista de dispositivos médicos habilitados con inteligencia artificial (Artificial Intelligence-Enabled Medical Devices) — <a href="https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-enabled-medical-devices">fda.gov</a>.</li>
<li>Instituto de Salud Pública de Chile, Dispositivos Médicos — <a href="https://www.ispch.cl">ispch.cl</a>.</li>
<li>Biblioteca del Congreso Nacional de Chile, Ley Chile — <a href="https://www.bcn.cl/leychile">bcn.cl/leychile</a>; textos refundidos de las Leyes 19.628, 20.120, 20.584 y 21.719 (esta última publicada el 13 de diciembre de 2024, vigente desde el 1 de diciembre de 2026).</li>
<li>Guía de la OMS sobre ética y gobernanza de la IA para la salud (2021) — <a href="https://www.who.int/publications/i/item/9789240029200">who.int/publications/i/item/9789240029200</a>.</li>
<li>Reglamento (UE) 2024/1689 (Ley de Inteligencia Artificial), Diario Oficial de la Unión Europea — <a href="https://eur-lex.europa.eu/eli/reg/2024/1689/oj">eur-lex.europa.eu/eli/reg/2024/1689/oj</a>.</li>
</ul>
